# Grouped conv2d forward and backward with unfold and fold

Most convolution layers can be written as one big matrix multiply: `F.unfold` copies every receptive field into a column (im2col), and `F.fold` scatters columns back, summing overlaps. Use the pair to write a grouped 2-D convolution **and its backward pass by hand**.

Write `conv2d_unfold(x, w, b, stride, padding, groups, gy)`:

- `x`: float64 `(N, C, H, W)`.
- `w`: float64 `(O, C // groups, kh, kw)`, the same layout as `nn.Conv2d.weight`. `b`: float64 `(O,)`.
- `stride`, `padding`: ints, applied to both spatial axes (zero padding). Dilation is 1.
- `groups`: int dividing both `C` and `O`. Input channels `[g*C/G, (g+1)*C/G)` feed only output channels `[g*O/G, (g+1)*O/G)`.
- `gy`: float64 `(N, O, Ho, Wo)`, the upstream gradient $\partial L / \partial y$, where $H_o = \lfloor (H + 2p - k_h)/s \rfloor + 1$ (same for $W_o$).

Return a tuple `(y, dx, dw, db)` with the shapes of the output, `x`, `w` and `b`: the forward output (identical to `F.conv2d(x, w, b, stride, padding, groups=groups)`) and the gradients of $L = \sum y \odot g_y$.

Don't call `F.conv2d`, `conv_transpose2d`, `torch.nn.grad` or `backward()`; derive the backward with `unfold`, `fold` and matrix products. Rows or columns of `x` that no window covers (when the stride doesn't divide evenly) get zero gradient.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import torch.nn.functional as F

def conv2d_unfold(x, w, b, stride, padding, groups, gy):
    """Return (y, dx, dw, db) for a grouped conv2d, using F.unfold / F.fold."""
    N, C, H, W = x.shape
    O, Cg, kh, kw = w.shape
    cols = F.unfold(x, (kh, kw), padding=padding, stride=stride)    # (N, C*kh*kw, L)
    # TODO: split cols and w by group, multiply, reshape to (N, O, Ho, Wo); then the backward pass
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0], [1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [0.5], "dtype": "float64"}, 1, 0, 1, {"__nd__": [[[[1.0, 1.0], [1.0, 1.0]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[8.5, 12.5], [20.5, 24.5]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 2.0, 1.0], [2.0, 4.0, 2.0], [1.0, 2.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[8.0, 12.0], [20.0, 24.0]]]], "dtype": "float64"}, {"__nd__": [4.0], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0]], [[8.0, 9.0, 10.0, 11.0], [12.0, 13.0, 14.0, 15.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0]], [[-1.0]]], [[[2.0]], [[0.0]]]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 2, 0, 1, {"__nd__": [[[[1.0, 2.0]], [[0.0, 1.0]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[-8.0, -8.0]], [[0.0, 4.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 0.0, 4.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[-1.0, 0.0, -2.0, 0.0], [0.0, 0.0, 0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[4.0]], [[28.0]]], [[[2.0]], [[10.0]]]], "dtype": "float64"}, {"__nd__": [3.0, 1.0], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.03419276725318417, 1.3597475403099617, 1.2247210785859324, -0.5103070767876675, -0.2979695111064471], [-0.5273841930334252, 0.5697263575719601, -0.056064439045617594, 0.7468856162565439, -1.8473247989741095], [1.5665487746995206, -0.09643216015562055, 0.6803784532741461, -0.13656633397682774, -0.3790985670748533], [0.46311015859758675, 0.824513527530113, -0.20252987069345152, -0.15278617857019708, 0.685698610809258], [-0.8703406419471712, -1.5143835037313955, 0.39498186274953, -0.6705658236878794, -1.9203405901180286]], [[-0.8140536639453595, -0.467597558892747, -1.1932024774322612, -1.4924638840630338, 0.03663782694480509], [0.8972492567277476, -0.23313207796045685, -0.7435960295088448, 0.3849938087479083, 0.7172358071943838], [-0.3000105984884774, 0.5446678079208929, 1.0428754765829538, -0.20695643620832396, -0.8135155419815723], [0.3476505985155095, 0.24754574096284754, 1.0988127684144084, -1.284580778805345, -0.6616129303555477], [-0.8381669607156745, -1.7340148462328515, 0.1264345551969962, 0.527804212495524, -0.7387900314758065]], [[1.3856470744961586, 0.8219243366604353, 0.6273764788355353, 0.4017070914409699, 0.955669564448635], [-1.3319798395431022, 0.6139296582498643, 0.6027768335334479, -1.7677185771429749, 0.34703010205437973], [-0.2504213467099684, 0.7815226960616993, -0.4390621876376686, -0.01824085764910033, 0.3428515173176555], [-0.8762616887442077, 0.5985966481803844, -0.10496318852366823, 0.49248262367924284, -0.5218375063367878], [1.0862015432775176, 0.6052019784294742, -0.17802502471933673, 0.6319571570936101, 1.259755161358625]]], [[[1.7911755134979888, -1.5735763704402195, 0.8831318116225195, 0.4650685085133813, -0.09386078018634399], [-1.0066649349770713, 1.2571886134731436, -1.2617379934445705, 0.5669454657347489, 1.3018679962026896], [-1.5996692880514796, -0.30251784048326236, -1.3092168175162993, 0.24405410803590055, 1.5143751306746547], [2.0235560291721977, -1.7781144428835152, -0.5749490057210601, 0.7035450933309114, 1.5793726520932216], [0.4212106442782976, -0.7461519975907094, 0.2971315096374385, -0.01661920098083325, -0.2037407223881694]], [[-0.7344710985474203, 0.38725975594300555, 0.3078796263658112, -0.09298412757635025, -0.2216879397006701], [-1.284916570539673, -0.48617566714143334, 1.2064496713466424, -0.19055825686602823, -1.4396973168381253], [1.3344428069513163, 0.5302654084829203, 2.1080870286253033, 0.06251216939171997, -0.46138457905169217], [-1.4476446987755465, 1.3238383700638794, 2.5695115129905117, -0.820934937223875, -0.6470876402241984], [0.5961887986723416, -0.8303454354133867, -0.27055836699218927, -0.34983769944206944, 0.19195888729205715]], [[1.09481844340068, 0.022067589713919693, 0.9189100957801388, -0.41988889600049356, 0.3277985357234032], [-2.138225589513189, -1.4499480667813884, 0.7959134126817742, -0.590149399040946, 0.5799149234726574], [0.5423442548146441, 1.3222788582368146, 0.8118590596762011, 1.0169913501666112, -0.11167133066420938], [-0.6982851765628781, -0.731558777725664, -0.4880439402887327, -1.1298291140131056, -0.5474435821203582], [-0.09256905150847415, 0.2516119571058145, -0.33889079006978534, -1.923682866353282, -0.07228295450292373]]]], "dtype": "float64"}, {"__nd__": [[[[0.22534578702721786, 1.084475594643844, 0.5778638956158266], [-0.6435606861455095, -0.7237765601247179, 2.0105975223757233], [0.756637954017235, 1.831439822737841, 2.129444531219653]], [[-0.818083666605393, 0.38526930227745887, 0.4581670048370468], [0.5596122240618859, 0.5419022252820672, 0.20205463691158088], [0.17412924424976314, -1.5024999410944402, -0.1654011831842485]], [[-0.7476197010175331, 0.12608730560642875, -0.46754626006357575], [0.6185207511629426, 0.8190757096799433, 0.30869236500853114], [0.31616814431588613, 0.09294668494996648, -0.4477978929622047]]], [[[-0.16450126423277087, -0.49564606330714495, 0.3879871341457924], [0.014113411841972533, 0.5812911986088503, -1.3286483115981003], [0.8877789284048715, -0.7626962368507467, -0.7342821862702664]], [[-0.19744298486173803, -0.5633845877352657, 0.2911251954982905], [-0.5741647764650408, -1.0694775505189256, -0.8458028083790133], [1.3119833252675124, 0.044258328637274344, -1.1675302849446099]], [[0.008377403177420764, -1.555946414705409, 1.7895665579723081], [-1.5244781602909778, 0.4787527046455685, 0.5436256643142272], [-1.4500077538405294, 0.30027651841828845, 0.9971720401657596]]], [[[0.46757528576185525, 0.261016274083352, 0.9490966814610048], [0.16090454888866232, 0.3364252880770561, -0.12671154648393168], [0.6318057260154758, -0.9413841472373661, 0.7917632588742233]], [[-0.5053015337885142, -1.090653373060846, 0.3652489557719693], [1.6929555515236436, 0.9616782115907366, -0.5156498276026669], [0.698059671998619, -0.454209119721369, -0.12401945613631439]], [[0.08952849321707658, -2.317218216900625, 0.19161476616248335], [-1.029609188818006, -0.6973986134552436, -1.4743904107207184], [-1.5165582181462036, -0.9432644007214415, 0.8255972952985817]]], [[[1.6660413474635754, -0.025205103519519486, 1.091790933563312], [-0.26398064687868195, -1.9119728247927603, 0.14991661602143025], [0.4456482631425715, -0.42861655475758254, 0.30222423421292294]], [[0.5725562476093207, -0.8634075676336961, -1.4768340975643026], [-0.22125416534201214, -0.21106054027038276, -0.35266321385541605], [0.9872467528013661, 1.7255887894094382, -0.4166470179838092]], [[0.6988867031627518, 0.9409311129269357, 0.7131564204233815], [1.0474862871297028, -0.38924397003268574, 0.7398063595133139], [2.024481935859489, 0.8036655593800595, -0.6198936689169603]]]], "dtype": "float64"}, {"__nd__": [0.6077124939345443, 1.2623802917054945, 0.3689862863896283, -0.5634006961428569], "dtype": "float64"}, 1, 1, 1, {"__nd__": [[[[1.5430805830215768, -1.2497670465629258, 0.5034250466559266, -0.016175574467128036, -1.1936657251775022], [1.2194496835829487, 0.3478875937013457, -1.166798177099069, 0.6017283565618432, -0.43160042891134], [-1.9027885021671262, -0.6889051253999523, 0.2635514475016157, 0.6357028862761235, 0.16556490512827252], [0.04516501126799043, 0.46999000714326233, -0.210976101965214, 1.176954603942294, 0.14864164821343293], [0.29392155247372515, 0.540798991545746, -1.0571688403044779, -0.7092268763497593, 1.5285791140398155]], [[0.33530869036097966, -0.2826160786151785, 0.3414835288114959, -0.48162223506155016, 0.27856628863229466], [-0.6658710549900579, 0.27601881733916134, -1.5777025523190094, 1.330919793665682, -0.5129578726969883], [-1.5812411969951867, -0.2238416945870532, -0.3709961947182103, 0.15239841860045156, -1.145250608848122], [0.3974655997902583, 3.623567688368005, -1.2786677207432902, 0.3377155756992855, -0.31071885365042484], [0.19260829481441374, -1.811809037837263, -1.1510093879783454, 0.4596595352026752, -0.038418019395222595]], [[1.6352381805909817, -0.7047973922713666, 0.17449952715639466, 2.913121452537828, -0.765561436417932], [-0.9038334934759573, -0.035130084895088134, -0.04456267883565894, 0.8441672801225347, 0.12957587279758478], [-0.7567393122486353, 0.21142155200757654, 2.6385967525787044, 1.2717736740410226, -2.7821691176232233], [-0.14201691359430452, -0.8750905219509657, 0.6222625144051509, -0.17326348062410027, 1.932695747995194], [0.9312355636189067, 0.8904028924592736, 0.18657106002655185, -0.04029765330752769, 0.34017992887452764]], [[1.295279791418775, 0.534389769872797, -0.3511325658376627, 1.2750846584688966, 0.19211036905670717], [-0.0776150634716236, -0.7913768386620321, -0.5659922764661505, -0.636046779726964, -2.79235715975389], [1.0168462003981633, 0.5664775694278109, -0.21427059157909795, 1.045983738409203, 0.43798777953339196], [0.37156747896960074, -2.4747635495786637, -0.23301591961061885, 0.5321891257672531, 1.6096868317882371], [1.8402948816174964, 1.4258382457202734, -1.1616095185495017, -2.197671367087202, 0.5704695747069322]]], [[[0.027615835350643075, 0.9625532543524465, -0.12588436360404337, 0.2197549635234998, -1.499989013056637], [0.44800965515897445, -0.44489845287418683, 0.349985357704251, 0.4594059935421338, -0.1172221448808959], [0.31680112211393313, -0.013607030533302129, 0.759199501128509, 0.5017429484459656, -0.5472163690372069], [-0.5879018811026014, 1.7437387779906113, -0.10159549531612938, 0.9918077625655023, 0.39338573028374196], [-0.5641124904961297, -0.7489737969543596, 0.12012568949433819, 0.2566544207133919, 0.6143637194600546]], [[-1.2669334275073838, -0.5946832824261059, -0.0948662034142554, 1.6007494601475956, -2.360178570626117], [0.43911164810792375, -0.540045858613423, 0.43137734284316537, -0.49197890628019864, 0.36455106356938455], [1.5646380406865834, 0.3541965034570923, 0.4333999424778123, 0.37285788843488454, -0.7503135205031676], [0.5430178518609218, -0.2083465627309807, -0.8117136441585762, -0.03854792621162633, -0.22728847344793018], [0.7534797953242457, -0.042839224997523975, -0.5294420574942039, 1.45597157572164, 0.9883877626799242]], [[1.2697294379343214, 0.6076673620145931, -0.1096989944346554, -0.419093403898315, -0.2793399950521628], [0.947690609812321, 0.11190328024474155, -1.4828431443678827, -0.03908279771702316, -1.416104206201203], [0.16586583406235444, -1.2038196342115577, -1.2639361282142565, 0.11857945392169375, -0.07821432580871404], [-1.6617616997896087, 0.7482237180713578, 2.163096688484254, -1.4813757442910636, 1.819849718777903], [-0.6877894242282072, -0.494090619164775, -0.17888722093774223, -1.4452772206653441, 1.2991272144382633]], [[0.49269362843733516, -0.7514542018131687, -0.27707715462169724, -1.5163594379661103, -0.34775774018957095], [0.9362131514399774, -1.7152897721430407, -0.43624032895121256, 0.14679173564781178, 1.048239345847918], [1.3493305527405735, -0.4572529558366793, -0.7403068765758416, 0.4503996611125889, -0.3632547919216609], [-0.31001937339348457, 1.080183678943012, 1.6897378703352741, 0.2658675711343094, -0.19221477473105453], [0.5712619649762886, -1.8875158294293481, -0.018249063351038427, -1.0605768454545814, 2.389427838640195]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[2.708975599026216, 3.3583879645993555, 1.7301448894639964, -4.4312447135363495, -2.9877188802378267], [4.40906769015448, 3.0206834826733826, 1.1699004268217914, -5.747945829828694, 2.290944233909803], [0.4959080608333313, 2.580091382093439, 0.36348939799439484, 3.116415703171805, 2.1923061056369595], [-0.4267603548746465, 2.165006317912808, -1.8431078214792214, -5.3930132254878185, -4.025140335478945], [-0.6995512636315654, 4.763030889352906, -1.6042975848613121, -3.1457921476178936, 4.818089289145689]], [[2.359963626761629, 4.915185610383006, 1.4840875788920773, 1.2669214869245309, 7.021120357684428], [-1.7885556934958504, 3.941323941575723, -2.4935036173261054, 8.364065612370586, 1.0837915269566183], [4.648011628169969, -0.3571656178144469, -1.6584478257328803, 6.661651508311583, -1.2652583414827527], [5.755623525137083, -2.5073577704624244, -4.378959530160705, 6.823138560812233, 3.270309639540546], [8.381815154063403, -0.9673275337618958, 2.0244553961330634, 3.664417091533439, 1.3911389249716197]], [[-0.17953143026600876, -1.6115898651346428, -3.808819852636883, -7.321486545085037, 1.1225154090923046], [-0.5115586479710394, 2.887351604432201, -0.9103705903284149, 3.543803973035008, 1.2064137095004992], [3.253685315689807, -1.6798493303027737, 2.530794250270749, 6.530911632059945, -4.798601712304579], [1.6202070715112886, 0.030242635941019946, -2.5006807685565144, 1.105557018441916, -2.4046732059428866], [1.4662110409472322, -6.0506993538787235, -5.387606298970184, -0.6921581275599071, 1.000421526105992]], [[0.5731118795767004, -2.973029318595943, 0.10815203533369133, 0.1307260793868974, -0.04514275193128203], [4.031995755762026, 4.199155900452451, 8.062308988125354, 3.072756898089494, -1.2685728256020226], [-4.164556095949081, -2.186161542366293, 4.196706060410061, -5.7460777251575985, -1.94864945517398], [-1.3811180177098674, -3.313466141555759, -1.7780286592992738, 2.6724433834370194, 2.0468137747961306], [1.5121644636631366, 2.6665603555853217, 3.29513748995829, 4.095518126705104, 2.913111701729298]]], [[[0.026271265864305615, 1.203430033738741, 0.7794443625755283, 3.4443148037388167, 5.023503945989711], [-4.282422074811871, -11.875729450521787, -2.459599246920946, 6.894310295332659, 2.846090805580835], [4.303692848891429, -0.46539657725216876, 2.801986229373935, 11.152072326412705, 5.17830756159886], [-9.146298097060336, -2.1769440263776296, 3.220970187796662, 1.570091790426691, -3.6014339808203433], [0.4295769259875878, 3.0723429741309443, -0.6315133487130719, -2.781270873429066, 2.494356525729939]], [[3.655901503729382, -2.7954294530706045, 3.8937713572784753, 0.3501804999502145, 2.5284285251967686], [-1.291884674201319, 8.859639656263026, -0.6349578354333882, 0.018700030092282827, 2.0683802803945244], [-0.6480452012458084, 2.5692698858129637, -4.880652961789238, 1.2257024540473958, -0.009044528526884088], [7.543749102505385, -0.27304054912133213, -6.445654351233, -2.5857993904544285, 6.9110227769217785], [1.941214425068125, 2.1641651412375653, -1.4491533769601814, 2.3507222462134525, 4.507719668226721]], [[2.875706042220712, -1.8880484982379608, 3.8373101693971066, 0.7245241699189333, 0.2468724829943288], [1.07346563110794, -1.6194922517018733, -1.8108273305209364, 2.3183463399458373, -3.37639944949837], [2.617659426810359, 4.694021848413122, -2.6449950931238666, 7.776475867106576, 1.3938009823734892], [-3.063970503108855, -4.142813979813442, 0.5036424612585136, 5.843594930671614, 4.712793211903312], [3.681220163669341, 2.6195634165832136, -1.5570529383480725, 4.584515134329859, 4.950327848513221]], [[-6.6382174872593875, -6.071364967322563, -1.5061317135345427, 1.526169568017436, -4.71325591308653], [3.3891772037241052, 2.2366338377058446, 5.128961253805216, 7.757385776920202, -2.0753725016213327], [-1.634381978691517, -8.154933147779689, 8.891449372274081, 2.051251937520194, -5.169840262460943], [-5.1633115483859555, -4.058188000010242, -0.42351557722503475, -2.4784530454182505, -6.8995126341346005], [-4.967697663766466, -4.012892936897584, -5.804727697921939, 1.8554827555004865, -2.2032077828019654]]]], "dtype": "float64"}, {"__nd__": [[[[-2.185861356522792, 0.7148164414628089, -3.0862094964606035, -6.96600461706934, 1.6916772115599596], [-2.9487280963204205, 8.525902629187634, 4.868281653651192, 2.238667652767399, 8.672189655667692], [-2.8741948566215387, -2.16768986628889, 0.6253512278172495, -1.446706259126957, 1.9750726867111812], [0.9967845938960815, 5.767095405085078, -8.093246870512392, -2.6184638058881022, 4.558685074003995], [-2.2420242199655007, -3.597229827939278, 5.332186074823586, 7.614589665162287, -4.184213347377373]], [[1.051525717283662, -0.16465739599897233, 6.054629369683845, -1.5547800222652342, 1.0338751422673642], [0.23831716365366207, -1.9060967482039917, -0.07556236045639547, 3.3049458607977966, 4.591678835848358], [-4.722122594002927, 2.550156084886572, 11.67669046365094, -7.275321039483108, -11.700291148924347], [-0.5046105097163238, -0.5036796804504649, -1.0959983295812463, 2.707537192195367, 5.415737039067885], [5.432516840731414, -4.341947975556613, -1.0209601465488714, 5.29751605450598, 2.2134147983969874]], [[3.120266714840115, -5.005814567224471, 1.377280743902498, -11.982455779245074, -4.379160527245592], [8.108906581286059, 1.7789517498351448, -11.972552820871517, -6.57956326448007, 11.209367435993327], [-5.091261487031591, -9.298010911870389, -2.6064773162243102, -7.055851011070548, -1.9378813635593084], [-4.962830812116734, 3.521670207027232, -4.456281688726894, 1.9879651742020408, 1.6915374526936706], [-5.591943358803346, -1.355788305974241, -1.3115438327443005, 0.792336241513822, -2.033253390704483]]], [[[-3.927862655933385, 3.182071606379385, 1.2459061208759268, 4.5658922172900995, -2.080680512190585], [-2.739575702732861, 7.26102188970831, 2.6303973721117475, -6.451671803205628, -1.806051014057465], [-2.976653286966807, 4.397178865259099, 4.3375986562482005, 5.459716637278707, 3.3996676362245344], [-7.2394613294121, -4.985504313692754, -3.082952730262228, 1.7506848413591742, 2.2986836789999403], [2.717725397963256, 2.3358244832152795, 0.16975524192940483, 9.253339271178621, -5.085112490446166]], [[2.0773516621798223, 2.9252959321759744, 3.389083891162157, 0.7512587496501244, 1.18249883863815], [-1.4550662429664847, -5.50960577821049, 2.740088246614823, -6.896292781282144, -0.7771875747700787], [-5.242057430903099, -10.469557678436912, -3.979481111717675, -0.9732164992650327, 1.130009478982139], [0.11326796085302027, 3.020817171496026, 1.9489607252933383, 1.500481862792178, 0.40683079275216694], [0.039260962296430835, -0.17357523625368956, -0.8234779483986401, -0.08348594018618805, -1.8449195608351434]], [[-4.374594693795603, -2.285066069943356, -4.713669378526111, 6.050513077972286, 1.0934830063489362], [-6.191313892780316, 2.429696886931943, -2.1509136915665157, 6.057708410210197, 3.640834633041641], [-0.23167729816041788, 4.792816379880822, 3.8218825231912787, 9.511056666339895, -3.259883351431067], [4.708542340196555, 6.165661701029241, -0.22561958537493304, -1.5567080592207971, 0.38635840064501037], [1.6218547296576413, 4.3360399593612025, -1.4510727424481926, 1.3370359402884824, -2.4158231400114047]]]], "dtype": "float64"}, {"__nd__": [[[[-5.567243018416354, 4.260093974246052, 1.999929085341731], [2.6785981561892114, -13.827946907516006, 6.263608754265189], [-2.030661944204631, -5.142704089315367, 0.16076056405845662]], [[3.9836719252197446, -3.294728684811181, 3.2155377738804956], [7.4162186489313395, 1.3642227791721733, -0.7599262514032987], [2.21107289692999, -0.5453201846955191, -1.8123038010444283]], [[1.0905564949707385, 4.8154222633012305, -0.3932005137782399], [-0.9461766676560563, -2.648788867154304, 0.9122137144019548], [2.069579439817257, -4.222064732301441, -0.15831336610678406]]], [[[1.6636311426812278, 2.763811166442702, 3.678757208413118], [3.9175997231528568, 0.877949731173481, -5.5526769148543185], [-0.7001474372306583, -11.615398314644894, -0.15995818130699369]], [[-0.9132086508304746, -9.492416461169299, 5.419252939144963], [6.548453328852483, 5.941410940219552, 3.6955700605013853], [5.171651639289568, -1.7587256624456566, -3.103377848647122]], [[-4.008400831237447, -4.322213535387863, -5.6322227257385435], [-6.581030732960881, -2.622561416505122, 3.9467918804015762], [4.917444983167387, 5.251198136455575, -0.12019216608250019]]], [[[1.7232300426769733, 9.627028614274849, -6.399024089067746], [3.520874499017152, -4.310623469650485, -1.1348851172285044], [-1.3473483969926294, -4.54269992669973, -4.152783842330798]], [[-7.0810696261765305, -5.2112633658964205, 2.029290861834819], [-8.436042265129167, 1.0582526305293989, -6.473078694328972], [0.9212331270318868, -4.0951171876755925, -2.873505956403556]], [[16.592104208952367, -0.6030480911646787, -0.8941756756655632], [1.255555576563875, -1.7337701386638613, 1.7975480515898998], [-1.0839980671338025, -3.955762199431643, -3.30553186798963]]], [[[-11.09836760691271, 10.149746017124633, -7.020141070731617], [-4.604533999753887, -3.5242944648137664, 10.190674731527032], [4.0146870167358575, 4.084435663398379, -3.0157332067901392]], [[7.188419345911818, -1.2693693837814677, 0.8473527759288415], [-1.9216699488879017, -7.407338563477911, -7.27131889240362], [1.2221521557541801, -2.131601050369122, -2.9410020109150983]], [[-2.949875498800413, -8.884528811028995, 3.7160881429259303], [4.696824241681304, 3.819832112554935, -3.2025600396067047], [-0.4963891991611415, -6.92351970778194, -3.1848449924443334]]]], "dtype": "float64"}, {"__nd__": [4.271112726623928, -2.3624490602515182, 4.508698672746765, 1.5649322377089243], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.7201999274777301, 0.14136984774843156, -0.19436462925611991, -2.5149560405578164, 0.7128045242850151, 1.629368948143988, -0.7288198970816447], [0.6834045108873033, 0.7225431565597581, -1.5481276661378405, 0.9357403471823039, 1.3547810757074732, 0.2284801183115689, -0.0631819947053648], [0.7879634903939723, -1.540753975374614, 0.0932807028593647, 1.0543419572150396, -1.2424353315034766, 0.9925712091532602, -0.4182296589367618], [-2.080826953612437, -0.10574795422979402, -0.9718966095973122, 0.31925964053282635, -1.1721411508347122, 1.0502759086725666, -0.738830321868627], [0.3563574951593681, -0.5939799434324009, -0.32290531215810686, 1.191760044649294, 0.21527844622988399, -0.3950320911956994, -1.2515080145573771], [1.1599512739010769, 0.73281116928691, -0.3265355739495917, 0.2433110221744552, -1.0295775519649915, -1.0148448852027037, -0.8261877084779022]], [[0.3534272179749535, 0.2958458940984602, -0.7453029308727064, 0.7377745962954552, 0.7846935409017799, 0.17290816429487849, 1.0383885677553186], [-0.2873624506076901, -0.11302113528190731, 0.4728586749266347, -0.03404400927754098, 0.7938991984511349, -1.2526681331938885, 1.4257650596774967], [-0.7812714862536482, -0.16469784997693218, -1.3775426990866404, 0.6002223198389302, 1.2738513731964936, 1.0710469286187827, 0.8333749722621981], [-1.5571432699572934, -0.7821680542782973, -0.46256299124016287, -0.9347342671990608, 1.1226952309586755, 0.18448431293126794, -0.024477804225176678], [0.48753573210350665, -0.9516190378282656, -0.9083507282671915, -0.8814619900328382, 0.4976209489663707, 0.34159637910150503, 0.578092467772177], [1.0783831175295941, -1.357300859038901, -0.5604943862405843, 0.14753809387269404, -1.3487431071780402, -0.5321752283329708, 1.0140289029037572]], [[1.102644129537584, 1.0767068924137455, 0.043017686109188366, -0.21196753704309648, 1.0795246901756155, -0.37264599644308644, 0.3382389901752798], [1.3252789241165568, -0.36757080036685263, -0.610552300932309, -0.009371273325325004, 0.598379178802324, 0.4824090158347269, 0.15585777321063388], [0.5564201353403723, -1.5995987439226662, -0.8592659923170728, 0.16322611442277465, -0.9286615229113686, -1.3416744423430094, -1.207819854394346], [-0.9181639232251739, -0.1784748908243617, -1.0226258159916404, 0.23489883850099222, 1.6962241426658793, -0.14762630914216496, 1.2746989107510511], [0.2884317418889116, -1.1198234993567935, -0.2564095749886225, -0.9818347415256182, 0.7679353842571606, -1.1530954140201652, 1.3225727009196622], [0.5781879425706238, -2.0223101003813904, 0.7843414146030919, 1.5530163621398, -0.7905237116146346, 0.985252212889632, -1.7983817317012476]], [[0.2521566116460024, -1.6613893616867887, 0.4823909896483297, -1.7879627500825985, -2.6328702074086703, 0.3960931685849598, 2.287889768374534], [-0.5956695756416828, 0.830165546635623, 0.11071109639467032, -0.32350473962469367, 0.6153842851040173, 1.6424053792567757, -1.2351279871889438], [0.5638969440170685, 0.3451354299335683, -1.4243456706601318, 1.6147108672323978, 1.9493587108638235, -0.952771988683857, 1.737900495331739], [0.056146118138976554, -0.1632483388444223, 1.5034489452647608, 0.22743285521144363, 0.3231759063686238, 1.5176749746362916, 1.2882199194979769], [0.893149417916258, -1.400549883940593, -1.5723903412447418, 1.1261412366905617, -0.9267195256430032, 0.14848354558181387, -1.074071995054169], [-0.18368652872814428, -1.2786951262487853, 1.6301740519090593, -1.01891242988674, -0.12199426763800637, -0.1881994816131177, 1.2060977932700745]]]], "dtype": "float64"}, {"__nd__": [[[[-1.4198975560984484, -0.033342032421233916, -0.20923426757877228], [0.5299255412154721, 0.47201677918099494, -0.4637465339414775], [0.2395859911253061, -0.3746311822420878, 0.46811523739189304]], [[-0.10258016582465135, -0.6555690937277523, -0.02755702261563491], [0.36378396130952134, 2.2658826013152176, -0.4912318707858116], [0.3066580644393553, 1.4630367216161897, -2.078168676007704]]], [[[-1.7430381976346998, -1.1886593501158287, -0.11423654283491436], [2.439976781493071, 0.3913669723953283, 1.208075341680204], [-0.22203935069643535, -2.4637237642990812, -2.3136383594407204]], [[1.0186192356728636, 0.1848595425053006, -1.2984991678126807], [1.1173206819999817, -1.5389549755475773, 1.1342421232028645], [1.451340849665974, -0.7648233755563159, -1.1081170380655065]]], [[[0.8443270506317625, -1.0952885786306175, 0.8618864571976392], [-1.195238777963637, 0.8075785602641581, 0.09386694355339925], [-0.43277837666164637, -0.8956846826243018, 0.23757476985481285]], [[-1.8926606333342528, -0.2759641722489213, -0.39615416922780433], [0.6783599277889846, -1.2538203551751494, 1.7134381385954434], [-0.31158077278064505, -1.1381675222664627, 1.3860787495037976]]], [[[1.6629310372409845, -0.1877719999739882, -0.8910728393162252], [0.016052968645974074, -0.690411734841456, -0.31203353977523773], [1.5584293956790127, 0.39190098723383365, 0.14539382420037475]], [[0.3460121962194285, -0.05808980630250469, 0.841567862220092], [0.879290130895175, -0.36971672115570664, 0.10508839204188516], [2.413607678811649, 0.4215991461724016, -0.1858533833881879]]], [[[2.049691752795071, 0.009432553838472772, -1.0973019243764386], [0.674580859823949, -0.8998261496510739, -0.49722251447843724], [0.32632538632402314, -0.12676692484843896, -0.7634866074150394]], [[-0.5509565875585889, -0.5933760026769729, 1.1661864544428544], [-0.7210002108727575, -0.57221906070492, 2.038354369963076], [-0.5537643267840531, 1.4894177085203448, 1.3157322025559548]]], [[[-0.1527516209306077, -1.4193913341554476, 0.45301413108094896], [0.672675147079706, 0.9678044817389633, 0.07189423769521508], [1.204557485284827, 1.390358607052709, -1.0543820408548004]], [[1.429488307035732, 0.17454235416077094, -0.5657440559728364], [-0.3186277280937623, 1.8775721415187479, 0.9840299776087483], [-1.7890459595244346, 0.08345750907388076, -0.13984994857922226]]]], "dtype": "float64"}, {"__nd__": [-0.17558727195693855, -0.5131552267403655, -1.4563014745345437, 0.8616443127058319, -0.9396897400963132, -0.3700411091760067], "dtype": "float64"}, 2, 1, 2, {"__nd__": [[[[-0.6023852857471396, -0.7278238778739177, 0.7432366755605775, 0.383774158464328], [-0.6741616336774211, -0.9448106915079663, -0.4694144108422612, 0.9670241777817157], [0.1259213756679188, 0.2330836920524692, -0.32816366806371156, -0.25788041379988563]], [[0.21654372665961266, -1.8434250101170515, 1.6120581122318396, -0.7876435517578195], [-0.5683143850602634, 1.3384970207154903, 0.22243372526484878, 0.04208355418804248], [-0.583319297100715, -1.5611693962115905, 0.18494535517551278, 1.1952613887247565]], [[-0.8996552067076425, 1.3627548617886078, -1.1271421287480032, -0.7340111257202147], [0.5058915495223177, 6.522835018385862e-05, 0.6908880011182751, -0.7417380918296624], [-1.031761620517733, -1.8670627011441454, -1.2037112496510525, -1.0761077546379645]], [[-1.426965620522486, 0.3084611753401447, -0.6638851654366239, -1.988295061993457], [2.48348264194843, -0.8348536612712331, -1.2555093274200142, 1.239788235820621], [1.1654338288257031, -1.1513409293575485, -1.4739818874941362, -1.0000667503395795]], [[1.1294857934675577, -0.31126076729256474, -0.8086978583645333, -0.028372412744297814], [1.2041914949024952, -1.3895503719058286, -0.3838017805531719, -1.3834756199231464], [2.17215549732985, 1.8183195343683949, 0.9511620791333857, -0.13340158457867954]], [[0.6591266158144133, -2.10201351100015, 0.2764267749739571, -1.3361652154299244], [-1.3247728086749904, -0.7047259124820933, -0.35021674668863756, 0.5742192448239287], [0.7006787483195271, 0.26267257681422196, -0.9443365940643537, 0.6759867880136667]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[0.6509590401113091, 0.9493460415534621, 3.611912340217611, 4.539825969902165], [-0.6836072202530027, -4.866522430999152, 1.9906959654934693, 1.8516312898032994], [7.281014626295136, -3.7729105998059906, -0.7524642031809515, 0.3757104255015369]], [[-3.279256367336493, 0.032617393812161555, -7.932075566987796, -1.0311837199284395], [5.577093998387721, 2.0784942216282087, -0.8642265066281599, 2.1971592995734515], [-3.5816406231870643, -1.3054880955519756, 7.037106680220998, -2.5309548485926117]], [[-1.0676806141224517, 1.1270656020882248, -2.1022827637816026, -6.451766791286804], [2.2578752982793366, 6.879287504918303, -2.7064544156431705, -0.8816311400267531], [-4.512113967077226, 2.6149724236105696, 1.7915651596623017, -0.943448673110371]], [[-0.4429093145860702, 0.38530668265906354, -0.8644683427977536, 4.380822317223529], [1.2845593521956853, 1.4511729498784205, 4.6122221546595235, 6.082368386380714], [0.8180986236295232, -4.299313064886349, 3.3427379152191645, 1.308041478631417]], [[-5.680321176733096, -3.6252118150291137, 4.543752416042989, -5.701645585032936], [1.5947318044159693, 3.402559177477094, -0.036475912368088115, 0.621319855086606], [-4.530129108431604, 1.9091015257255113, 0.40329356897232416, -1.8411304087039855]], [[1.6774584368681869, -2.656182447421679, -2.7661273822627406, 1.632478401087919], [-2.2302716258542157, -2.652451074861722, 1.2437683854037098, 1.9492182760315184], [4.5515753430236074, -1.7711165230318988, -3.7385624979339918, -2.19831259354234]]]], "dtype": "float64"}, {"__nd__": [[[[-0.9261301562775255, -6.0559168007392845, 0.03553086158381927, 3.9128955777804793, 0.07147066692940657, 0.6558711471420215, -0.7198814780939369], [0.6418886651116462, -1.7010202389609659, 2.034170335150307, 5.462984084926333, -4.246010008677147, -4.469504199057053, 3.184356590403496], [-0.13208791401252482, 2.4387043916902886, 0.07792970403335947, 1.5233663610233013, 0.42342807255548826, 2.0529453859920195, -0.12609002931599847], [3.018858494770808, 0.5607243793387856, 0.949109771994916, -6.352274935375233, 0.11854002245033868, -3.642620463118839, -0.03510805153908256], [-1.0020834691067921, -2.3140519062398606, -2.0087705345022506, -0.4532750491142348, -1.0546086518970026, 4.328582326912775, -0.522979602589064], [2.314096620222606, 2.3739240780623607, 5.431269185341941, 3.6787760733200305, 0.7454318049079681, -0.7289499346527781, -1.8843306109689195]], [[-0.5701793730198159, -2.4000089289261997, -0.5208651551522563, 1.9085915397781354, 0.6164359433207468, -1.7062889186909203, 3.002053241787403], [0.17432209120673978, -1.541993028843969, -0.33917721089181896, 5.618002546791107, 1.2955116273697274, -4.8920404598861715, 1.577833032178519], [-1.2872579879623365, 1.7052451170059193, -4.200798541843407, 2.528847693237477, -2.2722028735083994, 1.5623221837721855, 3.056404882125134], [-1.0331054904309236, 7.467122635375601, -2.332237899257571, 5.704632679726436, -1.0617352668593059, 5.801823068463321, 2.9138086913824166], [2.476668910359307, -5.417416886330901, 5.271670911527265, -5.813625553165095, 0.48103474616469, -1.1798183652782885, -1.0745343969800953], [1.8046813976685974, -2.6579737484170467, 3.660058976585767, -0.799486759484726, 0.7484590226233324, 0.7995448294301173, -0.06666146838339056]], [[0.606758649821169, -1.6879483131268078, -1.9672227340493154, -0.46285103291755625, 1.4535686606202776, -0.32073194465546706, 0.10512579820682724], [1.6394128301919222, -12.180916320872752, -1.6182717816252796, 0.6616885413734988, 0.9558951164599931, -3.9684037615896846, -3.6942559667724693], [-4.080309617098511, -2.89374210507259, 1.1447096290175025, 0.3861102521884008, 0.8732319142527659, 0.03031646342434957, 0.9446551521239954], [-2.2141658411038887, -3.0971824425228327, -1.2703468779015565, -2.028766950274711, 0.695818751083907, 0.4542689499477365, 0.6866602801755951], [-2.0810714763253206, -0.00850986739170767, -0.5870564799436009, -0.5432255898108008, -0.7521593072965838, 0.2677755419351747, 1.4647190973039448], [1.1555719238883366, -3.112263344510859, -0.31650534461130225, -4.9568338030350265, -2.0111973601901427, -0.7326166265595481, 0.5648478108578516]], [[1.1188175221906078, 3.966340864894878, -3.8826159198060437, -2.7592419473449734, 1.22721298735755, -2.7482596277217413, -1.7574054185768444], [0.04564539451967242, 10.049724805917117, 0.24103867058109185, -4.4699289006334, -1.2217690353423643, -2.6660855424905647, -0.1429071415689076], [-4.094602904959952, 1.9042707786439295, -0.21938717377859487, -4.329247955388689, 0.026244279203231986, 0.6457703300484008, 1.4114187354709038], [1.4957094063048202, 3.4161799350975945, -3.4466277061918174, -5.146797479729998, -1.7737912086081984, 3.605323079777326, -1.2347172099414885], [-0.3582542543055247, 2.8325188597676316, -0.12162039011592704, 2.162910728893129, -1.772377402244542, -0.14390291110739484, 1.715290290505752], [3.7850696735319964, -1.7123538834122, 2.244754960429206, 0.17479536424175013, 0.716436159225258, -1.8917826587244926, -0.5639017969802608]]]], "dtype": "float64"}, {"__nd__": [[[[-1.3012338288511365, 0.3915403584037128, -1.7620243262492652], [0.1461793102177258, -0.26116099711051394, 2.8655493760580577], [1.5756351808793034, 4.475472112369044, -1.1878284435340558]], [[-1.0117357290628324, 0.08703941625501166, 0.31947609055635146], [1.2883245815582123, 2.8847091061099412, -1.9828829625214799], [0.7048505632096431, 2.0885523028247497, 0.5238025476693218]]], [[[2.6643676277490567, -0.5306584633260697, 0.6501857740795598], [-6.708677229247442, 0.46064407854171907, 8.213911880435358], [-2.341825101470348, 3.480691307142428, -1.47414226427801]], [[1.057161328141782, 2.8416541567931324, 1.6896798605755459], [2.196824548874635, 2.732828597625365, 2.112388235851806], [1.4032133090626213, 0.9459846699395132, -2.2836919182756543]]], [[[-0.8400019082850481, 7.495983893675842, -1.2277563266002534], [1.9230541701918153, -0.00797682218513307, -6.621353627111423], [-1.3652321725353054, -3.978487033266388, 1.0509791103483486]], [[3.292605654386004, 0.49078243089576623, 1.4075205816105563], [1.535075048178046, -3.141818183937172, 3.4174533155041775], [2.95038836570874, -0.5662733410038662, 2.964459628903018]]], [[[0.9236035487913762, -0.4246669789391191, -1.7715518961580234], [4.570558977919315, -3.004862737559386, -2.253786092461538], [-2.341341541028995, -2.0278604002666376, -5.849807030141492]], [[0.0842776446467064, -7.305889535280052, -2.419407746925707], [-3.8055747330213587, 4.576018023697752, -0.8861100485452171], [2.0998818597224274, -0.1102695119439473, -4.915243913679006]]], [[[-0.23445018769918588, -0.41151972125067093, -0.7157238302052324], [1.0468482201719624, 4.955148019397841, -5.369137426255826], [-1.8610854462143116, 1.2694990863664937, -1.9187921691154952]], [[-3.5845953317576993, 3.5926957652344114, 2.321346023990664], [0.6753213891961871, 0.04744489757038961, -3.956045674780516], [-5.2726141322047475, -2.8093904977938395, -6.194194448631264]]], [[[0.17082972220300413, -3.0975679456130507, 0.4006611106953616], [-1.6706721528252189, 0.2865192641080535, 3.5725601790762593], [-1.247547431651076, 4.3309993084800595, -1.9060904154879825]], [[1.239632583316812, 0.7862417472542605, -2.9348367227655006], [-0.21801799661277643, -3.698744305075476, 0.6855211866729394], [-2.6229990000851218, 1.9177512251087332, 0.21980310829099833]]]], "dtype": "float64"}, {"__nd__": [-1.5515999019852935, -0.5320487572873367, -6.121590238177033, -4.59773252190018, 2.836754003839462, -3.6131200395804344], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.12094698222415633, 1.1802288531594096, -0.9171306451212585, -2.763225086388524, -0.47670920435042635], [2.1512634939784516, 0.46898735308559564, -0.19432917067429187, 0.8836363033330058, 0.5031824638654618], [0.46134130344171065, -0.5135456219172588, 0.07617103969374608, 0.2011920041925533, -0.6426617736770143], [-1.1187301636203146, 0.1390226610410074, 0.9684501383959119, 0.6473043244443822, 0.49397758966636646], [0.504314939922515, 2.5840027162940022, 0.25988372151032535, 1.0420380870646446, 1.9570999749725793]], [[0.05306310684670804, 0.2645300484882088, 1.0440465122729572, 1.6876632709395436, 0.3131053048715309], [0.3630178198307916, 0.0908395763866075, -1.2055991913108135, 1.1176678651447152, -0.2524987544235025], [-1.7187525937601318, -0.14413671467097858, -1.727029860446229, -1.0046959546512573, 0.0210839842272019], [-0.9169257795689443, 1.0753227232917877, -1.4641393051201623, 0.26913287834979754, 0.5642681004713006], [1.467471704389781, 0.9299717703254154, 0.24120445283023215, 0.08867354909967969, -0.05003194727880665]], [[1.1474753245159799, -0.6477984780045039, 0.5302385977171965, -0.08412698087139053, 0.2503985407752395], [-0.25238638051859014, 2.1453747242798897, -0.6467331644864861, 0.26380847414351544, 0.28688782084437325], [0.6192367647758478, 0.4617531850459596, 0.4360343563754495, 0.9934980811484962, -1.4932654754390196], [0.021734239849973837, -1.221681561296693, 0.24459577811774602, -1.1706072134118364, -0.9478177259625263], [-0.04824624693552761, 1.4088457450675202, 0.5899841756446085, 0.30932169109866664, 1.4161464455539161]], [[-0.9322607302853791, -2.0790228600623597, -0.41070043560170244, -0.2649597977310922, 1.248037662181626], [2.1525477747522057, -1.3163338419679362, 2.490906431956795, 1.5780632124495695, 0.8522765534536958], [0.38180803501383476, 2.4398859075439776, 0.09437491738358142, -0.4598529641182851, 0.15297851473621163], [-1.355331811815856, 0.48598403760958664, -0.09581363835408174, 0.34459910399179017, 0.37423523264609515], [-1.0447909066931365, 1.3303216392054162, -1.000670866316123, 0.9023624353218949, -1.5790757359239338]]], [[[0.38587727382680187, -0.8055641521426755, 0.05724462428210075, -0.6367143260963386, -2.168475081769634], [-1.099768727215913, 0.08372883459774622, 0.6391448233508122, -1.4539361819853163, -2.109847242467476], [-0.8568490922447701, -0.19603924778231666, -0.6178174625916948, -0.7942455642986826, 0.8660274310384468], [0.9813162749311736, 0.7571935711188085, -0.9225445507829415, 0.09168804514012756, -0.17823215261918224], [-1.4120382138249234, 0.8979007849930246, 0.10336226305510757, -0.4695143533298437, 1.4329354626707616]], [[0.7792069850166151, 0.8713792260973728, 1.2153939464516215, 0.7331652551567572, -1.2500243511709286], [-0.37364959880915405, 1.0182842115880062, -0.7538829964103553, 0.051864935612521784, 0.4124680042209678], [1.4561649558228973, 1.313318096968901, 0.9128286964153371, -0.17215624236752008, 0.9290069448924019], [0.5420740821601429, -0.1479790187664939, 1.3713910407205256, 0.5081032676371856, -0.7426578474682369], [1.2794140055440122, -0.24214041336753075, 0.6064618314522637, -1.5118072827459679, -0.0070879885675119905]], [[-0.9341909579445224, -0.9610256810104785, -1.89919196071258, 2.1544808263694883, -0.13906276038331297], [0.24888040746776988, 1.118491805411025, -0.11150329015524188, 0.523630632942684, -1.3491207189656438], [0.20616845441203926, -0.4042889768234709, 0.5100397480312, 0.1903166078379741, -1.3764606997022228], [-1.8428354003862781, -1.0060802936047188, 0.7459077891649745, 0.43365984107768557, 0.41155203718173644], [-2.0111770591381246, -2.681530496848263, -0.6162865474618695, -0.32557496658079027, -0.5966591680514005]], [[0.36102436500230906, -0.16507138655310216, -1.18024283064416, -0.8745474987662207, 0.14811927321531115], [0.07739624964763714, -0.06981572328599991, 2.271570437069392, -0.2587801015756447, 0.12378431203356066], [-0.9926693951689429, -0.35602904249170175, 0.5269494375734382, 0.5694980900139389, -0.1553387778553815], [0.3967377373535124, 0.05658234540382731, -1.8801957384853676, -3.2744750252613533, 2.3111715675830524], [0.8309166106085122, -0.056763679670587215, -1.7380382561234844, -1.275476926728536, 1.355910475560656]]]], "dtype": "float64"}, {"__nd__": [[[[-2.788177396734619, -0.3243311751827612, 0.8685643707938588], [0.14323113956807265, 0.7547927960126413, -0.1312493323652416], [2.3934515844200646, -1.0279959857034178, -1.3622270260269171]]], [[[0.2745323457384551, 1.4295806440689964, -0.055891882049737746], [1.049004893695953, -0.5308372772821607, -1.185274456798664], [1.3999205889632227, 1.5687130317863218, 1.5331289389866234]]], [[[-0.438252549382261, -1.1552072496527401, -1.169372932977615], [-0.2246351948398268, 0.5824377910705448, 0.5091717251007895], [-1.4344057124813594, 0.19517282113137333, 0.017730241573760675]]], [[[0.0901277427551174, -0.023697257578161837, -0.20192024059231037], [0.5932751351544383, 0.12685956100129553, 0.22541081625333215], [-0.6907558179163049, 0.39577778077264153, -0.12115202407949936]]]], "dtype": "float64"}, {"__nd__": [-2.115161206719354, 0.48361008112032944, -0.2643015661031351, 0.9262818985828198], "dtype": "float64"}, 1, 1, 4, {"__nd__": [[[[-0.1620987660364429, -0.09574252299710637, 0.2116899379073339, -0.011856407610606527, -0.44692981301642565], [-1.7932005266838178, 1.8426882782588792, -1.5787861979761342, -0.3415219400245493, 0.6531383002851368], [0.3800170124042527, 0.5672684708544655, 0.7691897816929107, 1.2633727016452962, 0.29728336176131304], [2.9066426225656414, -1.0134117960896127, 0.22737890400828, -0.36085292235598465, -0.5451373075246495], [0.2872651697767749, 1.4582406713706946, -0.7102085890488052, -0.7109557180725844, 0.7244888389664226]], [[0.5439940928415591, 1.957756906252454, 1.0278799668439151, 0.22959129154398192, -0.32937303577863], [0.21526280165615264, -0.1752170241568787, -1.3650909850692166, 0.00823476571828589, -0.1751221845637408], [-0.23254078864950567, -0.42930876907218096, 1.6900694204184323, 0.6962661826895761, 0.9368627469435644], [-0.6076712288741689, -1.0226215177842206, -0.2828836517938628, 0.2440717813696992, -0.3453289296327535], [-0.18822795612362872, 0.9614195296530823, 0.299224116235975, -0.9840149322390461, -0.42799928293781514]], [[-0.6575399806434574, 0.18093838884366548, -1.1672517633310884, -0.17874735939774014, 0.6926417557428085], [0.028723962725316857, -0.9138764686153973, -0.6164140563073847, 0.12435099552344743, 0.48308154737365394], [0.837070338085314, -0.14521142628142528, -1.3172497998456858, 1.4249926150693548, 0.2978863050101901], [0.406071879762164, 0.30551662807689645, -1.037490328454873, -0.31930453296537076, 0.25725854377995006], [1.0326280972960145, 0.06018184941311413, -3.1199484617298703, -0.24135127104889734, 0.3526540938558663]], [[1.2938891287798602, 0.711316250719178, 2.490208765842763, 1.424342290872007, 1.347663807613821], [0.942787347199645, -1.3824352885511482, 0.8470504924830344, 0.5994841169964561, -0.5260205456504803], [0.5866459720818912, 0.4180661464478143, -0.46501958937202054, -0.12339128475917634, -0.2632948196828485], [0.23047909197918892, -0.4245753866565146, 0.5449131725222097, -1.9175878572894989, -0.2819282236523998], [-1.6846874347825345, -0.4828994055152588, 0.5169365582122687, 0.02161206220729548, -0.11063522406501289]]], [[[0.0687923704514405, 1.5695009171706567, -1.1062519664390633, 0.887733163892781, -1.5125967976357788], [-0.4674409530656326, 0.1635974663667906, -0.8516802796258283, 0.2051373748588469, 0.7286924130231177], [-1.1937425853510049, -0.07737187402361104, -0.37656624380573456, -1.940383750108247, -0.1870741593687321], [-0.7977706641384916, 1.3390848762278962, -0.8755627219446027, 1.6994721983325756, -0.5334468091161984], [1.9381440416467546, 0.5393889643174486, 0.45384663864571767, 0.8057140372434636, 0.9023076736227501]], [[1.4032914160112737, 1.7327082739402266, -0.34671371601491774, -1.250134840661803, -2.671756314672663], [-0.3872530433388884, 0.7602483316290657, 0.3783622610501528, 0.11518598504297725, -0.3501964891053096], [-0.6399925073998515, -0.2836743665965275, 0.38897601384866326, -1.7646073410001109, 0.7289218423079096], [-1.4584585195145676, -0.1582504219456442, -0.6345183311758219, -0.3695299930364056, -1.047759199579851], [-1.1557211984076157, -0.23337388951821267, -1.088032491064515, 2.4552684940445824, -0.4210308622095371]], [[-0.6616194265509461, -0.5554501579040428, -1.0777614685478931, -2.462776380293462, -0.26251774179364884], [0.3036781808383792, -1.5107386734307533, -0.0015084293193091856, -1.216866519157122, -0.04295723664760023], [0.3519157300894203, 0.47802792648450976, 0.4276074663440487, -0.7118833694527169, 0.8723000385221371], [-0.45599626107499447, 0.3419435745509745, 0.6964702375276849, -0.31238669103838407, -1.2928166248758384], [0.6050906711403551, -1.125698335088863, -0.899648652284072, -1.1112617052061289, 0.8338878305671076]], [[-0.1720209649202813, -2.2575388980458806, 0.7259496864877375, 2.0618931968003893, 1.7909938086046109], [-0.5771153198389466, 0.0680386477378159, 0.49938811388986465, 0.8838751739940897, 0.025115592684801253], [-1.1043341175393315, 0.7447573420627167, 0.6051027468781527, -1.3408242997700495, -0.37759608861586347], [0.2584249738264761, -0.013537040158141394, 0.5134997651079655, 0.7845666975151093, 0.5305721846024586], [-0.1261440721745425, -1.3604287739217926, 0.6042048753294543, 1.3895822171754864, 1.9639082719526526]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[-5.211712849855168, 3.8102647901522353, -2.1571329622115853, -6.328558283714885, -1.2730866006290666], [0.7366830292932987, -0.7413311435140424, -9.285434047599239, 2.348133462012316, 7.3923719559145775], [-1.029248015062701, -12.90548973361664, -4.179322105792408, -0.19616152893929195, -4.156868669716161], [-7.611924172660644, -5.1544680786451895, 4.6307054823672775, -5.503733251097671, -1.5199433021828361], [-1.5900673556953007, 3.788639524341926, -1.8251529348141187, -4.029383412565323, -2.453715560244767]], [[0.8506410645317204, -2.0362721951246217, -1.8440015991540017, -0.009726446283038481, 3.2563173131709204], [-2.6728999920081025, -2.7004424479553846, -3.086343520354826, -2.354773403770987, 1.327592500741316], [2.290930614531285, -0.7404112857115359, 0.30015823202369274, -0.3011706844565418, 0.626294972231964], [0.9745651609366067, 3.9879549095181837, 1.433723154599559, -3.37549041243358, 0.26636712339827306], [-2.7685715451723674, 2.6108061886295024, -0.5869244728211771, 0.7001206458159097, 1.4837403256459514]], [[0.06296980887946291, 0.13989184577710856, -3.0516724193391833, 0.6793378873438041, -0.4219777893574068], [0.24205459165250548, -0.45233086686992774, -1.7784709684095126, -0.7053212451198352, -2.1253854296806134], [-1.903126340356645, -1.7891942818682207, 1.669571154910618, -1.4968531461991217, -0.31010265845467283], [-2.1134340884962945, -1.8162519361338063, -4.211514645719185, -1.8370562441540659, 0.5689416829366096], [1.8284362138790327, 1.9832511310913032, 1.5420716677188404, 2.857841205058153, 2.098978607113844]], [[1.350788562321351, -2.292764401447247, 1.2849483761437712, -0.26896873647614394, 0.1746691266020831], [1.2000401178555453, 3.3364788010540067, -0.8994046164901965, 2.247915495390153, 2.2953633026345535], [1.1441913727605242, 2.3459814769694542, 1.3704472265017709, 1.1306597149993354, 0.7049801685469198], [-0.21249517998070466, 1.489228655783509, 0.16635049501453958, 2.225686449981815, -0.11514657752413537], [1.0275967026446737, 0.13531264167640256, 1.7684758659823214, -0.0012258819199608695, 1.2834998304049754]]], [[[-0.7016739077104608, -6.264415362955917, -0.5798131576391745, 3.5955552094783894, -5.154119257992516], [-2.6331970349067486, -4.065967441521231, 1.492389793690574, -6.5226432410306385, -4.228592105308002], [-4.347020280006537, 4.116619457095593, -1.5731243880852954, -8.119366723956173, 3.565543430946902], [-1.137796835867298, -3.809470244864726, 0.024455986335990332, -0.644248105622645, -2.8997509092417255], [-2.959406459775959, -5.436200898022873, -3.579256389517629, -0.2551399112612842, -1.298677209711577]], [[0.012166425801763803, -0.6836178459908495, 0.20591927666571003, 2.5093462704116485, 2.6359142694051343], [4.838038072818406, 7.33457721270014, 6.832827272297368, 3.0601248972786195, -0.05032954681044888], [-1.8136072476154947, 4.256543287814447, 3.502840619213358, 1.8538733605025202, -0.03995211703607715], [4.01535506610858, 4.072578623212669, -1.032123541582986, 0.9518448583193819, 0.5641420929544546], [0.8746617221942266, 1.0960543759979546, 3.5910678232113433, 3.0750907244509724, -2.020719090217513]], [[-1.2293309667946657, -1.7218751773838334, -1.6744311754932113, 1.5845872058045547, -1.8436796611816728], [2.686212114834146, 3.6493354841094363, 0.46469537897753393, -2.834000060415779, -2.492905680000487], [-2.323025707325978, 0.9030762184065022, 0.8431914032485399, -0.9252906407973454, -0.3214449096084693], [-2.0553801752064653, 2.0742922062456928, 3.7027274174396148, 2.0063690218742547, 1.735232209854122], [0.5042932793165631, -0.5905266172970252, -1.1145284955643686, -1.9284118549879425, -1.2041632457908338]], [[0.9739624471691205, 0.4971895160146771, 1.4601060530870398, -1.5380004812468486, 0.6539699000646993], [0.5953962492493796, 2.2310866747297347, 1.6898685737935382, 2.0336861330344256, 0.25126255402617925], [0.8825275421595642, -0.06294116069141975, 0.5158939370259767, 1.1847970440940723, 4.394761158913242], [1.4205145255442433, 0.17172675803150006, -0.6704811094746534, 0.513219697981564, 0.7495025229598608], [0.9980697505149491, 1.434336231763147, 1.095450260210507, -0.519556577368995, -0.008306110500888697]]]], "dtype": "float64"}, {"__nd__": [[[[-4.692215264032373, 2.2261148287548806, 3.235416697162372, -3.182324234796138, -0.8442501590430028], [-2.856979202513274, 0.22742955431131143, -4.877265976964638, -1.8734392706985057, 2.0143040366226463], [8.504729780857987, -0.5224986156974438, -0.9653480026929357, 6.794681099514975, -0.28424158577706193], [-1.1431682862198793, 1.383099174772977, 5.192486490419082, -4.421593172339268, -3.2431956705277587], [-4.987877863074807, -1.4122612728359898, -0.5462161010895066, -1.5831865741214746, 1.6921139868283201]], [[2.024557026204881, -1.2430614728819345, -4.564727599046401, -1.6457163499445004, -0.34809674980124505], [2.8457068347430474, 3.613245649090404, 8.507544792117818, 4.062978031104837, 1.218907427664072], [-1.3839677488328477, -1.0850664784248418, -2.3366979418578353, -3.44519898066823, -2.0919968715568587], [-1.7210946703916992, 3.769436041648159, 4.689935966201404, 3.2973977031757924, 1.8742950106993732], [-1.27639848600147, -2.9052206710918953, -4.000514606983078, -0.8155358440609781, 1.2259972573650164]], [[-0.056294701886330734, 1.3250631454784896, 1.1786813731439276, -0.48857750515848486, -0.39106436117714155], [-1.0692046968640567, 1.084976018900892, 0.24650816906587136, -1.6354175503962007, -1.5337716409682882], [1.233646139129432, 0.9707681617507344, -0.4949479100796192, 0.882085090190399, 1.0717544945961017], [-0.6797270129884839, 2.584007876591657, 0.9589152144482618, 2.828092229531714, -0.0544978902157066], [0.22894206430149253, 2.8166977922220178, -1.471377788995278, -2.2581078081861437, 0.127058357646769]], [[0.4392112049741742, 1.7780076401081408, 1.6343710752964367, 1.3088950426974728, 0.3834434646473154], [-0.6560387068884138, -1.2259608684126446, -0.01749852716390446, -0.6408769579650554, 0.4603674711946162], [1.6067816588557775, -1.3244406733814627, -0.04933952356380472, 0.13131530492873955, 0.0518495734042452], [-0.2828518867924762, 1.1351836095801955, -1.2264893351774024, -0.21318447540373675, -0.5590096317548959], [-0.11571449645890497, -0.7066840269396948, 1.56123715719266, -0.5765847789187918, 0.11157518019347]]], [[[-0.027807693693742502, 2.93274269221054, -1.0674739250691174, -2.2393968413374328, -1.3163742029486594], [3.9593118097509215, -4.253880302747567, 5.954084056109705, -1.8311060169012499, -0.7559372404891775], [-3.5148846141972356, -0.21152813781751922, -2.699720733855428, 1.4271029896378473, 0.7340497235657338], [-1.5008887991875344, 2.0374812172657717, -6.67008925171458, 0.9982656568241997, 2.617026267775194], [5.565294315226349, -2.1676943378509055, 3.5307108522388826, -1.1532958663956432, -1.1913798778203932]], [[0.7278033669137007, -1.7344227180474872, -2.6510539253797547, -1.6807363501926589, 2.392948657897084], [4.63728940027931, 4.573535408954762, -0.5311538062142489, -9.454148767485572, -4.917786723330864], [-1.629474451405284, 2.1268576246637796, -0.8006791102482098, 0.7303909223675771, -0.1453602522455814], [-2.5091528269929713, -0.3023246354354231, -3.0266062559984683, 2.1529845928035103, -1.306844432974063], [-2.140751367681602, -3.0201507695317957, 1.6744608588655319, -3.4746589647270665, -4.896838790729051]], [[0.050695336130444746, 0.9724743855459956, 1.9443320441608687, -0.4978859172672594, 0.06571893001076079], [0.5678169144790808, -0.45028246363926977, 2.0746860994698864, -0.8830339099689233, -0.9147460759260332], [2.7007815778939164, -0.09277162753081025, 1.3032444673790113, -0.45574123814362205, 1.9743978805314681], [-1.1650685839738923, 0.28381528750976226, 4.605649193909488, 1.0508072859883042, -0.418250242093592], [0.025815245348359023, -1.0858291595764902, -0.2574505899268861, 0.513165721255589, -0.3379966128212308]], [[-1.341355978820131, 0.26544977703582373, 0.8605791032327187, 1.3682414969343455, 0.5129102583780507], [1.5517720822416208, -0.9394038991689638, -0.5459409163887651, -0.3938827144213024, 0.9411417551913233], [0.01899913307348558, -0.049145089809538314, -0.9107004159340536, -0.060209318054241345, -0.6182740487738052], [-1.0463869898373543, 0.4839186967276093, 1.9886144371247323, 0.2089744172647517, -0.06996660581368405], [-0.711481494451543, -0.23392722484635137, 0.2573262089828607, 1.359420756816734, 0.6773042413836974]]]], "dtype": "float64"}, {"__nd__": [[[[-6.6579004124825305, 11.08450524150517, 8.790980490045689], [6.489064602679671, 0.2444714283701201, -6.7765328926027895], [1.0000435487247414, -1.9647945358804855, 12.925273371461532]]], [[[6.680135839357271, -4.56600541885524, -5.404127767408931], [-3.39921831659341, -0.8107031977514337, 5.474903056242699], [1.149960900352855, -4.488701340699518, 3.0496400887059707]]], [[[5.571663444674853, -1.79707203428712, 6.825145383961423], [4.543944775541921, -6.655522073925803, 3.2043649668891705], [-8.09916133046866, -3.2402116339445293, -0.22961365953810983]]], [[[-10.866783008367687, 0.9423857638426413, 2.964405285739862], [-23.190827251743976, 1.553953775285954, 2.7956824161732503], [13.962599683391911, 8.652835207616121, 0.5884097231308759]]]], "dtype": "float64"}, {"__nd__": [5.199484875237997, -4.052807591876361, -12.021354464664592, 10.433253863645492], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.045246283116937853, -0.04780886952570091, 1.434280343550896, 1.4265428153589603, 0.32809247121433227, 0.700002626465831], [-0.7743433618140189, 0.5709987234848108, -0.8394697920019472, -0.15864133262379854, -0.13671644398724112, -0.482520095630155], [-2.125680935482724, -0.8402491540892708, 0.31922249107749323, -0.07493984446773366, -0.4077997729778312, 0.4145791813491706], [0.6745865076768218, 0.8333793506260436, -0.821886467639888, 0.00623406170298788, -0.3497337522178352, -0.8942205459170451], [-1.5127911505986935, -0.11460391500807023, -0.889385046649373, 2.136916506597393, -0.1152332459411415, -0.24113189194574752], [-0.4143143399457627, 1.8589039887604408, -0.9585285259673488, -0.1776804116011882, -0.7863461058975564, -0.14136204354355847], [-0.05249828991261809, 1.0822009059673012, -1.4329453261908138, -0.037632824117134145, 1.5996841350692554, -0.5265783549171041]], [[-0.19427301748741202, -0.9037776239918246, -2.6351055336003513, -0.1214481608057499, 1.1588192785966591, -0.48466804587942536], [-1.863440071819456, 0.2495400635915037, 0.06212856579081349, 0.9835210036072138, 0.8935942839550007, -0.2871144194711792], [0.38293658147100573, -1.8163076030037426, 1.096862391849496, 0.16897082232891666, 1.0682864034105377, 1.0457680339736728], [-0.30434552864159387, -0.16643412157405474, -0.8917707127170484, -2.0833543861034, -0.14897192308259663, 0.13664513578363807], [-1.1028067827601193, 0.6948064276672696, 0.08146980625651742, 0.43661718575424435, -0.7866363456082952, -1.7815136455477132], [0.7609203505444048, 0.594005219049885, -0.126806956633467, -0.43810644531985704, -0.2940190916131104, -0.15393824210675439], [-0.525822949366172, -2.027963030255994, 0.7672681884231806, -1.511756711303726, -0.34980647912782614, -0.32361113732897717]]]], "dtype": "float64"}, {"__nd__": [[[[0.24438486174758725, -1.6300730032774473], [1.190809085504232, 0.6398502055341888]], [[0.7547039080463267, 1.2526731473541355], [-0.8122609013482742, -3.148609230721722]]], [[[0.3202567142063565, 1.3008009875237423], [1.151605750849984, -0.5454813193525592]], [[-0.1007293599673798, 0.27729627896711423], [-1.8154452656612146, -0.6744498675661362]]]], "dtype": "float64"}, {"__nd__": [-0.6432456781360035, 1.5771997316747841], "dtype": "float64"}, 3, 0, 1, {"__nd__": [[[[-0.6571857760556928, 1.5657506815022777], [0.009061643193566853, -1.6804941364081298]], [[2.2144748612210705, -0.9744051072455392], [0.6167340173233123, 1.7055869870144302]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[-1.6839738988098918, -3.3583149508592642], [-5.441716328943034, 2.762531520282579]], [[3.280938093339678, 0.29808322553131594], [2.715654795524708, 3.554442912940074]]]], "dtype": "float64"}, {"__nd__": [[[[0.548594187723386, 3.951851878009194, 0.0, 0.07058598587777588, -3.819795041533303, 0.0], [1.7676191923036364, -1.6284551228553252, 0.0, 0.7423796120057968, 1.5333656788586973, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.19972773834631596, 0.7874770788384833, 0.0, 0.13553835706062917, 4.957957360940956, 0.0], [0.7210251281388371, -0.33061879119924326, 0.0, -0.036983902950674254, -2.0056303585269286, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[-0.71904330893634, -0.2091733356052353, 0.0, 1.2798293611577576, 1.691174923723601, 0.0], [-3.4864515919173056, 0.5756689239088842, 0.0, 0.49718107894130753, -4.272749653249378, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [-0.055284365303581884, 0.18236932521567986, 0.0, -1.4400781777668077, -1.6321569540107084, 0.0], [-1.1270072703899319, -0.44448774971201044, 0.0, -1.7314001387993656, 4.140886432553958, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[2.2589821571221944, 1.1404076256475457], [-3.3442893644665967, -0.3967058102287909]], [[3.43582288253416, 2.6572101510103416], [2.0208491293099, 2.56338553295709]]], [[[-1.0635541800878603, -0.5080944611446939], [1.1515242383472726, 1.1304590625595006]], [[-4.0529153760956484, -3.4872824097614155], [-5.020338950568044, -1.2312885915850806]]]], "dtype": "float64"}, {"__nd__": [-0.7628675877679781, 3.562390758313274], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.5351468856747267, -1.230575462309461, 1.0409207199531214, 0.06541732204974776], [-1.22733274654892, -1.081201651314647, -0.3896613630724056, 0.9640196909012788], [-0.287058031402476, 0.6545200654761228, 1.6527138753238326, 1.269425049267331], [1.43757869116865, 1.2319057135585894, -1.6085270583188667, 1.2229435218257196]], [[0.32854055273811855, -0.12439786387837073, -1.0846491000801854, -1.5890076930151398], [-1.1110998630375328, -0.02136399482727128, -1.17508790770756, -0.8164748147369453], [-0.6167085805248366, 0.6665386451776084, -0.2951798056789239, -0.14589427107571593], [1.3262297809776586, -0.32124297387233025, 0.5992422671630798, -1.057297666586972]], [[0.3291463422989192, 0.11951478122981696, 0.175342375278584, -0.46637535592930623], [-1.0305983919227464, -0.2995292240944625, -0.4679740939722483, 1.0333753457626227], [0.9078343846283474, -1.2021212081408479, 0.6273230775466384, -0.47973155597128947], [0.45892034462216985, -0.4815574861963912, -1.4717675090881714, 1.0162674929083935]], [[0.4522953836127022, -1.2256871105721943, -1.89141209206216, -0.5342238407490858], [-1.1818436200084754, -0.3639146373504742, 0.6202767049951368, 1.4690849181260075], [-0.35680157231134757, 1.377353060310801, 1.1643443115472718, -0.4344197167399898], [-1.6670231582692812, 0.8535907181430092, 0.610407333067874, 2.3549212757476528]], [[1.4317753105174584, 0.7843567297973222, 1.6907226407742137, -1.045271336072919], [0.1969992459275695, -0.26955280958951944, 1.7466891519725556, 0.5698139010319786], [-3.054409785875796, 1.3762550612628786, -0.9003005306016896, 0.784834898279859], [-0.47039437835630715, 0.2542203923061352, 0.645869233266703, 0.20539789936058755]], [[-0.9420003426071543, 1.411979037793882, 0.41041830314563366, -0.0028246695832332422], [0.72066109144984, -0.5203461108747987, 1.9015866631546194, 0.2410086038599166], [0.38215729866985937, 0.36410595194621, -0.8954163404365566, -1.1567473348264072], [0.2351377851521245, 0.37067346267737145, 1.3645829477980946, 0.47005983513517285]]]], "dtype": "float64"}, {"__nd__": [[[[-1.0703065070589624]], [[0.5454326269158963]]], [[[0.8466376651878754]], [[1.0223782301362037]]], [[[0.3530128584270115]], [[-1.2022851992986199]]]], "dtype": "float64"}, {"__nd__": [1.1807027778667105, -0.7126451574996581, 0.5930529003397351], "dtype": "float64"}, 1, 0, 3, {"__nd__": [[[[-1.553756661657519, 0.7839295138314688, -0.7658334629642978, 1.1015575232646482], [1.7831123843812566, 1.1884467944280226, -1.149410569982895, -0.9258216517301068], [-0.8526060764198489, 0.6804380419217609, 0.363208259059973, 0.1758833350681741], [0.326613190365866, 0.5967998336589533, 0.6682053642997745, 1.7062328287385546]], [[-0.039956010876422, 0.21919611037047668, -1.8408836836562135, -1.3093453450228516], [-0.6796089083133181, -1.1731341329496865, 0.23316038800365207, 1.0529566952548757], [-0.424633101377309, -0.4754325742819696, -0.5024564480003911, 0.13940193965006484], [-0.537044212550725, 0.4339412434556159, 0.30755662830701436, 0.3503631473016489]], [[0.7694076921851474, 0.03608632659775611, 0.1478251752685504, 1.7630028582840247], [-0.042849661754502735, 0.8003554235427914, 0.0557023261143775, -0.4571846069305433], [-0.45680872944342554, -0.8204005082150312, 0.011350684604329225, -0.6048055301595403], [-0.4240492383736855, 1.2183550362541784, -0.13467800076143494, 1.138945390627345]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[1.9326707085650616, 2.4299450489257115, -0.5250044499703137, 0.24398955221167784], [1.888294885762114, 2.326267320891616, 0.9568285859545322, -0.29642577325053954], [1.1515698756630812, 0.8437176169671436, -0.7492083340230014, -0.25754660811256525], [0.3654219436323364, -0.3130303225556217, 3.2291660391036006, -0.704906275027541]], [[0.028439487046157952, -1.8645752609713635, -2.4979322454260267, -1.6536749247464257], [-2.79347976230554, -1.338296283297566, -0.47469225194108977, 1.6642097710129797], [-0.3088245337301326, -0.322230466359613, 1.0088704647017566, -1.562945223101039], [-2.0284340945689414, -0.24765729559097244, -1.3346317962493495, 2.555385425911616]], [[2.231041064981536, -0.8276605873075398, 0.6964598811734561, 0.22745473659364285], [-0.20384399670793962, 1.123501720146876, -1.0765928696334295, 0.5044434569900782], [-0.94465509297443, 0.641129436439323, 1.351781049905684, 2.260849911164695], [0.1442949573639199, 0.23714074974838711, -0.8195648368076542, 0.10041501734015207]]]], "dtype": "float64"}, {"__nd__": [[[[1.662995865358253, -0.8390448597293899, 0.8196765387341868, -1.1790041850499073], [-1.9084767878206808, -1.2720023373696776, 1.2302216123350433, 0.9909129382228097], [0.9125498315501752, -0.7282772639193198, -0.3887441630894464, -0.18824907800669852], [-0.34957622293987395, -0.638758745376884, -0.7151845494617531, -1.826192099156495]], [[-0.8474695775559341, 0.4275807340459995, -0.41771055748471475, 0.6008254136132056], [0.9725676718993362, 0.6482176570346526, -0.6269260265906681, -0.5049733355587662], [-0.46503917198613365, 0.3711331086588949, 0.1981056348566305, 0.095932509476963], [0.17814549040663602, 0.3255141010155729, 0.3644610071693195, 0.9306350539090105]], [[-0.03382826375863528, 0.18557968310232423, -1.558561463813152, -1.1085410858347602], [-0.5753824993752685, -0.9932195432727252, 0.1974023665137111, 0.8914727980145292], [-0.3595103775115713, -0.4025191246443479, -0.4253985539936443, 0.11802293270799201], [-0.4546818582166069, 0.36739100118798607, 0.2603890257029059, 0.2966306369993437]], [[-0.040850155683139225, 0.2241013313733079, -1.8820794023830545, -1.33864617648154], [-0.6948173528661676, -1.1993867985574702, 0.23837810482504435, 1.0765200025047457], [-0.4341356386433803, -0.4860719138434993, -0.5137005340271633, 0.14252150833698715], [-0.5490623115325014, 0.44365208046725607, 0.31443920131518355, 0.35820365444320984]], [[0.2716108087140091, 0.012738937302404578, 0.052184187669024946, 0.6223626784178351], [-0.015126481578587604, 0.28253575582240226, 0.01966363736266997, -0.1613920449213808], [-0.161259355335235, -0.2896119284599611, 0.004006937617277732, -0.21350412899408344], [-0.14969483375209192, 0.43009499392703276, -0.04754306601602938, 0.4020623679376283]], [[-0.9250474805407111, -0.04338605636553829, -0.17772802030910254, -2.1196322428360452], [0.05151751412239077, -0.9622554799038763, -0.06697008225382106, 0.5496662862597494], [0.5492143743202382, 0.9863553885239978, -0.01364676010169174, 0.7271487373647704], [0.5098281230705345, -1.4648102275793322, 0.16192136698660148, -1.369337185960642]]]], "dtype": "float64"}, {"__nd__": [[[[-1.0461685539836547]], [[-1.7434084906877771]]], [[[1.58362122702053]], [[7.945800081531959]]], [[[-0.41289351723410783]], [[-0.15009029605954016]]]], "dtype": "float64"}, {"__nd__": [4.126998646263785, -4.245918264685538, 3.000254637840337], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.26417665170547694, 0.8604647648752776, -0.6125485144108256, 0.14534966850868355, 2.6334357590583513, 0.8249422522909028], [1.0005831224157575, 0.23006220121732388, 0.8538740411102476, -1.1662077824303652, -0.8554574675757844, 0.4239656843995215], [0.8148446177972511, -0.41575700376833397, -0.19367209976426145, -0.3684451278803379, -0.19505501189423233, -1.2083534625897254], [-1.4414090149740613, -0.09110951493520601, -1.7477857041449045, -0.98940635578445, -0.6719051641186565, -1.0654547845029316], [0.22459286179347188, -0.46078431764569394, -0.09674201745452715, -1.0273183246386468, -0.7603642790277262, 0.49291114973973355], [0.3426088066876693, -0.4866865462495704, 0.3412599429685959, 0.16409028785188778, 0.13492742860949888, 1.5693581962091065]], [[-0.4270815018779688, 0.3923336793488718, 0.3936132043354695, -0.26269738006181825, -1.1942617779173776, 0.6606037358735629], [0.23064220816098716, 0.01739664428876653, -0.010022941568681795, -0.013103215393435803, -0.9295695279875268, 0.25536736674286714], [-1.8410225171059418, -0.06546352983029337, -1.536941593992995, -0.3940461155791132, 0.6720966100856779, -0.3036533713106844], [-0.5251619532876218, 0.46698211587146804, -0.3169370957782779, -0.45648328582932707, -0.44273563701106333, 1.46160993226029], [0.6132090632226698, 0.42897718996138373, -1.2427889402158996, 0.5656417378581519, -1.3473043961381836, 0.42419040769028843], [-1.1718221343183859, 1.895603717228975, -1.7412808024107558, 1.1635712893655976, 0.7422861755758774, -1.83441921985434]]], [[[-0.3003213848820837, 1.3842202036892637, 1.0756069659915977, -1.7189749793028049, -0.5809509236589622, -1.2348156834117328], [-1.0861672072226454, -0.7039632626171968, 1.7788757198303244, 0.07068753618358639, -0.28876287060255074, -0.397897512142318], [0.6026076255275586, 0.6273232976640245, -0.9829012320881564, 1.4031995195164957, -1.146831544639249, -0.341352480476096], [0.3860474275195959, 1.6459484311131196, -0.4108023073742853, 0.22112316754638975, 1.2375005510493817, 0.5583157802700378], [1.0620877486976303, 0.9731562176643914, 0.1948288088337314, 0.5656928876167495, 0.8514796765482758, -0.0928489874205226], [1.533063825331432, -2.0509927494562996, -0.9856200412836568, 1.378679437961256, 0.36198965384489745, -0.7088592267126451]], [[0.31323063489322445, 1.0218591576645937, -0.62350576497552, 0.7033435741199093, -0.18653243367243058, -0.17770590343991546], [-0.13341560119590715, 1.0473266051966845, -1.183518288333394, -0.4313532494798761, 0.5986196638376883, -1.2826531659587812], [-1.6444165981355792, 0.5052911362960196, -0.06201205297899131, -0.210647520622077, 0.07886789532900715, -1.035490397663295], [2.8491751613489824, -1.0473935432444814, -0.498635597095578, -1.4848307734587933, -0.4791674732412087, 1.4628460455439483], [-0.04441430580737882, 0.5314553905764877, -0.5467332524174038, 1.7496943645716212, 0.100539674192258, -1.3116220410590687], [0.7066411318586284, -0.43293975139833224, 0.5436133022424647, 0.07581271081316814, 0.20010790717662733, 0.07320125815863321]]], [[[-0.9688631855343154, -1.5655864097372054, 0.912246436293951, -0.1991193528657931, 0.22192807452489735, -0.5276110559905913], [-2.086988841268274, -1.340802616311021, 1.3313013950124823, -0.8494139233129958, -0.4469436036701316, -1.5316957779517222], [-0.5923236804163224, -0.4374735853596646, -0.2868009926726687, -0.13919707409038573, -0.6972665052145701, -0.6404386432745673], [-0.0801378159486739, 0.25611749203012757, -0.2089758211373433, 1.675453227235025, 0.13803770887918546, 0.09354804505006091], [-0.7238397375737835, 0.9589059609882956, -1.0919997444061282, -1.6868062445258347, -1.0442450160409973, 0.5497585544840727], [-1.2139903902944866, 1.2240608220955693, -0.09267057812811809, -1.5863130329377844, -0.2994648490070866, 1.7128576066546422]], [[-0.9307715122640611, -1.3806651879473577, 0.4413258541398111, 1.2350669236607366, 0.9449986041361611, -0.3999276131575112], [0.11613193904813295, 0.2534978858909002, 1.1024837029332017, 2.077085663155936, -0.17933948758995313, 1.227682192437719], [-0.41556275306242846, 0.517399157178201, -1.0048019376897126, -0.8356346380911165, 0.34358442806814304, -0.9683771260668537], [0.8154066562938312, -0.6222600347473726, 0.005838892072758227, -0.15516750577883512, -0.4050349568728924, 0.05931839672733896], [1.0076042713296924, 3.0335854821149075, -1.2584743962969978, 0.22851102788798622, -0.24714728016435158, -0.9848824913181701], [0.21473674555551744, -0.9502750101825099, -0.1497426029800441, 1.168145147531617, -2.0925251157568807, 0.17607127706863812]]]], "dtype": "float64"}, {"__nd__": [[[[0.3751587286931521, -1.0289368391540816, 0.4523115073548639, 0.2799926970804284, -0.4205060483409081], [1.1404625689154928, -1.3789129719326403, 0.5715424474029138, -0.3051298445373, -1.127952759398139], [-0.24913943121702004, -0.42086158976733773, 0.7472099746790617, -1.566830850897147, -0.26628277878438883], [-0.5273307665299065, 1.501106552187745, -1.1899964583818867, 0.9865162752936628, -1.8289299262502543], [0.2708164743150586, -0.20995900856050959, 0.30316494144597456, -0.7782979478466677, -2.0270565257994924]]], [[[-1.6222025092709764, 1.2343794906402028, 0.13416441325251965, -0.9441108539862885, 0.8190341801437231], [-0.8266993890375026, -0.8212763583607363, -1.2960744043818648, 1.0936036565380511, 1.8779273078759633], [-1.0492350112492796, 0.20318725332373674, 0.8908385309610426, 2.027793737515435, -2.4863354350836255], [0.05198819258348294, 0.5554592465165616, -1.5702325162579818, 0.3166602733141551, 0.10546060302293304], [0.6309510656694569, -0.6754188271454158, 0.32438088329322307, -0.6989731045956905, 0.5119256709824261]]], [[[0.06392409261931463, -0.47143476962268654, -1.510366553719727, 1.5529634264138805, 1.0138640868909958], [0.7845959934108626, -0.7597670316876737, -0.4457536435803356, 1.5747874896371499, 0.6491732288182619], [-0.9468332615698813, -0.04836178369741325, 0.047066684260107655, -0.2937443406010689, -1.037113293250514], [1.1618333282666622, 0.4969908122824552, -0.2888616060558032, -1.6640813124802076, -0.673837259775155], [-0.0938907377170931, 0.7793191736336359, -0.45747012050601754, 1.0215887392545346, 0.05443437100972997]]], [[[-0.8295054939625199, -2.8769414494992316, -1.2578526280251816, -1.5732849816584376, -0.23332638099728253], [0.2516428580339635, 1.3100728544510323, -0.09181019396009256, -1.2367612922804587, 2.274788651017388], [0.23447163241268845, -0.339828352518828, -0.33805475534881263, -1.4271226444466791, 0.5615082226730084], [0.30003145371750645, 1.5157936943683599, 1.029991994143079, 0.7971756247988485, 1.1879535964991175], [-0.35381345299631534, -0.5123939568573195, -0.11742368226939992, -1.3129259409249059, -0.20899314306443156]]]], "dtype": "float64"}, {"__nd__": [-0.39443160933547894, 0.604964681200518, -0.0775109269611645, -1.9703651835833618], "dtype": "float64"}, 1, 2, 2, {"__nd__": [[[[-0.030533712092280497, 1.3561124485333538, -1.894617554080115, -0.8176887831544787, -0.04048190764680544, -1.382924361671262], [-0.6051078791441458, 0.0009799775010914348, -1.373550419561734, 0.24097062613096298, 0.35691183598081316, -1.3197687267890887], [-0.023597822667560657, 1.0301981417589652, -0.22454010008409747, 0.3878889850481277, 0.5963341100396337, -0.2808763245835156], [0.9375659784052449, 0.3407048070692528, -1.0343896528159024, 0.4800287993011603, 0.8236763748705842, 0.7647067473104314], [-0.6830645328053451, 0.4216464325276245, 1.335646447337708, 0.19486944326861635, 1.1497427911075346, 1.2670500939403464], [-0.06901960948011329, 0.6289850236534812, -0.32119713144328527, 1.0383584545243214, -0.11974513039713343, -0.5781483207890694]], [[-0.05319877868814072, -0.0995137577877571, 0.3558129084855411, 0.43964207072340933, -1.1898699520586227, -0.6759539837505846], [0.3360838108613875, 0.4580120505477821, 0.06361461536398388, 2.6126878967317424, 1.0993094738053089, -0.09434194446782972], [0.7118019276865918, 1.3227681133260774, 0.35164834048589283, -1.757962420460769, 0.08635334519924182, 0.04128171441089483], [2.124765356817415, -1.8443561066137464, 1.5411441682628426, -0.15991496668512944, 0.9288603382146026, -0.29322405077912617], [0.17557393034071064, 1.13305330818012, 0.08690158678773251, 1.0578899402679587, -0.07623198018782656, -0.4042925229006534], [-0.2124540014286109, -0.49583494577315196, -0.11223287410048813, 1.3709280242797857, -1.1605408091090466, -1.6501662701833861]], [[-0.00733765837766603, 1.696622697823297, 1.0595718203727365, 1.3363866286170905, 1.265077278693292, -0.7137894893992427], [0.1348032662206669, -2.643703252163099, -0.5915677306839868, 0.6260487558265333, -1.5375753657406064, -2.0406415885488998], [-1.2780690222081161, 0.24098990561154054, 1.191337686751841, 2.612340377004637, 1.4580212245152326, -1.4869720925997094], [-2.2899967525935017, 0.18226093209354458, 1.093726799838784, -1.5820544124199443, -0.48443274933231834, -0.40964638472432374], [0.0765870730918286, -0.13500929873532694, 0.563395347114185, 1.106906897709879, 0.12307475061205052, 0.8036585593301802], [2.0483502264461357, -0.9651687354068543, -0.9892770997431434, 0.569436045436966, 1.9142934439907555, 0.23064163714144353]], [[1.1583209754681223, -0.2412367703607982, 0.9845798981775188, -0.5563954713638761, 0.7466516814827324, 0.46929364627852366], [0.4704445560315992, 1.454630347586232, 1.1479530672596694, -0.5164524836669299, 0.9699474017190732, -1.6122071125962536], [1.0784096663338012, 2.841458255092077, 0.6248390124470375, 0.6046737845652295, 0.16275290599842188, 0.934125201163783], [-0.346351955440302, 0.7786362296251339, -0.1614594135330178, -0.3116697355230143, 0.23903559129783275, 0.05693879723491398], [-1.6170666017329134, 2.196741837069548, 0.2615403984178413, -1.5731934244166577, 1.200948572671362, 0.09538653629960472], [0.32877430441403876, 0.8316703263788524, -0.7703133275757775, -2.8188073688078505, 0.02522033865479652, 0.36731309210116414]]], [[[1.127985981509672, -0.6316029675924437, -0.14142221983445924, 0.5246556851379028, -0.19098036796612733, -0.12410193619178779], [0.3802420044886951, -0.8707066369927025, 1.7093414799900337, -0.7972414252221333, 0.16097679449580735, -0.20489669157411625], [-0.583640680076424, 0.6592050300399567, 1.1348644254984965, 1.5793047490576846, 0.7051250529601226, -0.3255288081912007], [0.985080115328477, -1.3225916425426203, 0.17032915526445644, 0.6430517509520587, 0.6838953296610933, -1.0335420950681156], [-1.381098131148113, 1.7766578072630086, -0.39810124134884095, -1.5354465220706803, -1.1293165204115572, 0.12187366312015575], [1.2867186491836213, 0.10220125153689945, 1.303722302035366, 0.3278641806927512, 0.5716690739590594, 1.0827052803681734]], [[-0.2792740481452679, 0.3914216003318374, -0.2940448506435303, -0.5446316372132957, 0.7473875807547903, -0.7314819250157563], [0.16838142575392698, -0.7041932709518828, -0.0037294668736127514, -0.4435728265878499, 1.1763858608506348, 0.15966450415038158], [0.15541539414195818, 1.7243146722771285, -2.822103826101674, -1.0474823182965773, -2.675649842472509, 0.0543309361155401], [0.49710208837666736, 0.7001056809659153, 0.3225477327801375, -0.8200017417251237, 1.8029852002103743, 0.3530975594718368], [-0.2146913374921499, -1.07687830029312, 0.46296148482296173, -0.4504893593392732, -0.41084858631785726, 0.2444013010470319], [0.2538370544946887, -0.7863626183518988, 1.2866188889272856, -0.9701332699025007, -1.0861010571383831, 0.33748189941986967]], [[0.15581681206738568, 0.0972350893561588, -0.3853809564412033, 0.06759358580454475, -0.5387553158554321, -1.6251484595016648], [0.7581910348801167, -0.9910189717497944, -0.20723098458644054, 0.7522143548955657, 0.04475650152096843, 2.7624143438501796], [-1.379819663841364, -0.7292046938156375, 0.1067829991118769, 1.3744405573077645, -0.2313249327542024, -1.4148984469003536], [1.1594442862211718, -0.9062055691243516, -0.18343754702481962, 0.8957656670907848, -0.6558253411578755, -0.43880071772405255], [0.5000855262303783, -0.19197736183148612, 0.2977192716127737, -0.9359434494224875, -0.10908458992337573, -1.082509475845548], [-1.5489135566868197, 1.763744392497683, 0.46176435080758327, 0.9235436599403296, -0.9614047531530301, -0.0226431162328224]], [[-0.19979739597836188, -1.4154510837896817, 0.5515324776254494, -0.7170167955038498, -0.19801665896755663, -1.5124909143965162], [0.9974307795869707, 0.76359320381765, 0.3967170210293235, -0.9050182855963325, -0.5346614561930653, 1.226079134331416], [-0.2725732339575764, 0.27105903340285453, 0.14485576493480964, -1.1144986579867593, 1.5283166461098254, -1.9798102442903582], [0.22801780282570736, -0.08354317250515872, -1.613818447202866, 0.336073237101463, 0.5242025296995936, 0.05571998514633553], [-0.8236974454683194, 0.4540460850839588, 1.1187983026539592, 0.22526311556205536, 1.7280607533910304, 0.8633707079274557], [0.7738565590505608, 1.3948531284615238, -0.6567584554535372, -0.516515098347375, 0.520401752195974, 0.5243023025896619]]], [[[1.5143526571879582, 1.1509201728158354, 0.18134093871624685, 0.2252534808161542, -0.26781240915876314, -0.3561781047940635], [-0.4419983294613296, 0.9148128875968262, -0.45901741230591475, -0.34192065943782957, 0.17331182785875568, -0.497245448756211], [-0.21193507800945188, -2.1642742459534183, 0.39416565900242256, -0.283044236965684, 0.2165513811971711, 0.3956986053998324], [-1.8249333732352193, 2.437552948717358, 0.2015687867976168, 0.22977942592155864, -1.7245488984259443, 0.6920809176683177], [0.49139429095078907, -0.2715624686112965, 0.3450907176201157, 1.3890803398798206, -0.2198898717818997, -0.7813128627417485], [1.4605163433269583, 0.4003522436222269, -0.037999270313991376, 2.443525120795079, 0.30446485613588764, -0.02715979914998026]], [[0.4132464386978885, -0.2543710239479247, 0.36088571706959016, 0.2952294005397631, -0.5112581205695875, -0.11616485701609072], [0.4517004451636945, 0.5226628433384667, 0.7210120764227228, 0.9219391834505197, 0.38448405072656355, 0.6508653456452774], [-0.2481876660462902, -0.12627585921282775, 1.1667930620826013, -1.8491159591046427, 0.493716428168585, 3.2818068939356255], [-1.734318680876337, 1.6134370247603984, -1.6482142740060586, 0.46504254718353466, 1.7209543165945362, 1.5779813026729694], [-0.8211223185218014, -1.2193651505793228, -1.361498803018955, 0.7885647153539254, -0.323002951697789, 0.07014421892996499], [0.31975717602227177, 0.4688089222563843, 0.5914093375963374, 0.1775089055820223, -1.5537521595540862, 1.5059634573774334]], [[-2.697430357331017, 1.2834580929693309, -0.556522711827019, -0.1522937382631042, -0.47826029633057243, -0.3265678007533975], [0.020421764912572946, -0.23861097345319043, -1.0573861991153275, 0.8892454389143212, -0.5405218274418392, 1.3817715747541919], [-0.11449361562184061, -0.6676577078203444, 0.4747867779867456, 0.9258006484915726, -0.8862846144548489, -0.2461156560422948], [1.4645536859553416, -0.9611939962940466, -0.8127199645731206, 0.20388489052032968, -1.1775339224992258, -2.427327893268688], [-1.630703684746048, 0.4113347637755736, 0.6917589179896703, 0.629277624818383, 0.4275811896861688, -1.1959296382668625], [0.7144575386762667, 1.1891821837722374, -0.6604478856090648, 0.9798711906875789, -0.32944448770744217, 0.2280895878849682]], [[0.40578861568605123, -1.4474015792032107, 1.504409465036717, -0.8167674347740908, -0.06182324322575265, -0.4543454968284723], [0.4977503630091825, 0.4717484617014224, 0.06107362529602389, 1.4539319337837535, 0.3228421849044196, -0.35889062702867397], [-2.839308879274253, -0.24120304877857118, -0.24952559513014716, -0.8302331801846367, -0.8196539723955043, 0.6341829263026746], [1.085926041649664, 1.9357604555055592, 1.1834930351536779, -0.6665120630029823, 0.29734291096159804, -1.3222918094990759], [0.7718214149807961, 0.3988372907654232, -0.5694808791157911, -0.06422522616327209, -0.20915745048410772, -0.5952231655535434], [1.3621968196989271, 3.1466547522276036, 0.8654130475369116, 1.0998726683149544, 0.2022399052663044, -0.3074784009982932]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[-2.944334415087967, 5.862338648060506, -2.0619746564056785, -1.2417434270845895, 0.5137889482990495, -2.5213819128325734], [2.4959195757675285, 4.3211196489959605, -1.8966420761097451, 4.829516886954994, -2.1163924105006497, -1.53153914630813], [6.518748399846348, -0.9541337614803863, 5.989828726912469, -1.1824257485696603, 3.3230716884092453, -2.5016249615237585], [-0.6483460793648121, 3.9147535622678973, 2.2880165886247026, -3.5429777632467543, 2.198654031196611, -1.5629971488486805], [0.5181387129264974, 3.3809163914173985, -0.751526940852726, 1.3676824110531167, -0.6324422952321034, -2.496603061895178], [0.9688153612605652, 0.5031561231275591, 0.46230928903171165, -0.6501664220568495, -2.4252327940399363, 0.6773651582190212]], [[3.1556802978627267, -0.5748718374504469, -7.026618160259017, 4.354151708965515, 7.078106310403761, 0.02821036216051198], [-3.442075496215094, 6.5795820054896454, 4.008870166600637, 1.728235598213394, -2.7530287666157225, -0.06583468827832462], [2.6445536599438575, -2.388612111428177, -0.2176863631171816, 0.2981530046591624, 0.7999666605732734, 5.391825338538642], [2.3408866162950015, -2.21714358458009, -1.3564188620749296, 2.4990159040425084, -4.641317333943469, 2.752789608609181], [-1.7153611851258543, 2.0416728311512538, -1.210393618662715, -2.8611237425370586, 4.151012759430531, 2.532163929074678], [-3.440756306838213, -2.048951583716555, 0.589729303704774, -2.7919046426275718, 8.56072517130303, 3.325844205270261]], [[-0.01822526066690832, -2.7428235198013042, 3.028834125907192, 0.23783986597534196, -1.642799414261153, 0.40877905224286437], [3.3788110289454933, 1.4831371003999139, -2.783788927082509, -3.506532716971328, 1.5942891193853461, -0.6082384985757037], [2.6987598433970383, 0.09692623877761468, 0.25898274578760355, -4.413943739676941, 4.022632332328601, -2.090335180461969], [1.801875460611595, -2.5038449712414645, 4.00471647789856, -0.6526847236473886, -2.70650613634774, 0.9856259098962283], [1.3560995370257862, -1.7490563198124573, -1.1868783391428348, 3.6743230417051795, 3.475315846672621, 1.166014712298198], [1.9089023800173148, -4.286665077187675, -0.46074778042006076, 0.7964706673111921, 5.085900951376737, -2.0511300194382094]], [[-1.3020197948960228, 0.7281254557097598, -2.419779192849022, -0.271171196387719, -2.0391324229336623, -3.1686804790636947], [-5.888864292459702, -7.553179514284383, -5.518225782949454, 0.8762567565856612, -5.100120504603456, -2.630687194885814], [-3.2755965308172126, 1.1488226690271974, -4.932613908462646, 2.5322710934285526, -1.6754270808581508, 0.5275379577125576], [-8.68868717891046, -0.3483655218715971, -3.755660099892018, -4.477200852567569, -1.8252825034010935, -1.2600990789913504], [-3.7250600983592905, 6.449378414425248, 3.5996362697652833, 5.664324727540442, -2.464316931349877, -3.992307390280197], [-8.671320841566448, 6.041731666788176, -6.1661249381905305, -0.9036271166018943, -0.5461419098678666, -3.1774386832932633]]], [[[-4.042622973120899, -1.6441422972667812, 1.71270861829677, 4.5490693562140905, 0.03804044777736659, -0.12426569044623181], [-1.0158279146632072, -3.541852459425599, 3.2402541500192696, -5.152348244656728, 6.617643297108178, -4.538859087617093], [-2.193039997967646, 0.0039143602143906775, -4.983874061658175, -2.5159940026878154, 2.2802998932443224, 0.15219936642145793], [0.6485298514973106, -2.6073372149617424, -2.3678630407151298, 0.41825540681282625, -3.843180843782708, 3.7498940029338543], [-2.179219734435483, -0.23926515577435725, -7.568671314978473, 1.275822457137249, -1.103449784772744, 0.11372455021960537], [5.125644953569292, -2.8615719370061523, -6.2515426322305725, 3.444449882502128, 0.9436004871282364, -2.9397821582621573]], [[1.3940592484867498, 10.506770308922984, -5.060791494719515, 2.5807359927656135, -5.136511474541309, 3.2077943533239868], [-4.376860724153162, -0.5507100700292851, 0.5182074017787069, -2.8895621425274056, -0.3624726027855383, 3.072999378099169], [8.005747913746667, -5.646641992295515, 9.0105725890263, -4.140497687227813, -5.743804784238453, 0.7696239927392339], [5.430266040674179, -3.16048422574279, -0.17693724699549807, -1.28519198420911, -0.355057660401378, 1.8651419320537035], [-0.9088467715921673, 5.0025645489193264, -1.3170902502516562, -0.8417788811758438, 3.6670678522997737, -4.172911981844124], [-1.522779502777083, -5.92898106890475, 2.04035029878476, 1.481938453375684, -0.44820921944483394, -1.2023955278761307]], [[0.6420089137159949, -0.28706395421399483, 0.9214565393765852, -0.09651904008080528, -0.35792223665392525, -0.020998827554007483], [-0.8237815547989964, 0.9457302956891545, -3.5605662289787876, 1.8377615705161854, 2.37552559420262, 0.20471475789167765], [2.972540540422959, -1.3852155776164614, 10.340748164601422, -1.1309996132208895, -6.538190967484442, -1.622544281246882], [1.7588033148446238, -0.7192584324586369, -5.692598168915927, 1.5819533881841195, -1.92937297147328, 5.861312787338908], [0.44367486165511066, -5.92862639256242, -0.41001821385679055, 0.08810717401583402, 2.85750414809007, -1.5740692729827628], [-6.414475563476942, -2.339691190973725, 0.33321328430547487, 2.684040872063414, -0.8130483230920986, -0.35290238887493436]], [[-5.050511108458164, -0.8081967668141186, -1.8558682168730452, -4.570193538970146, -1.973634555749816, -2.0694362660838026], [-7.0151762859766, -0.7883842550589708, 0.9460512501874445, -3.751904281898528, -1.4809462854161617, -2.3034880848507098], [-7.072854120485449, -1.3791985485433738, -7.8077305377899275, -8.177529696792263, -1.5063586271940839, -0.9079738243298843], [-3.437312837628101, -3.198751213099338, 2.4011764564998783, -0.7483187547453201, 2.1565038620514656, -3.0673677608117904], [-0.8048671843702464, 6.0862313600289815, -3.564954692886065, 1.1687956481984225, -1.9613517146365924, -0.6347787033503296], [-5.00270457743654, -3.9964908288881347, 0.6878190394512396, -1.7942776583491888, 4.713437695820686, -0.5827376584034825]]], [[[0.5599213331455506, -0.7060691171050175, 0.2896736363289274, 7.072820763023575, -2.2630602600975798, 0.6818062808804479], [-0.3147501071972142, -5.52629905532354, 3.932387170722965, -2.003637255716427, 3.001004483018624, -1.7318959987349425], [-1.6427015084751546, 4.321970555662172, 7.407572908211488, -5.125591483409208, 5.486546729075778, -3.172924394095863], [0.11094350546842724, 6.487830927322174, 3.035542621158201, -5.88768142033864, -1.9739305913819993, -2.0946317613618577], [0.5166945102028568, 1.138370655280796, 2.1795618705777153, -0.7370927131999303, -4.297695123344466, 1.3742353784424153], [-2.545989487703715, 4.815986389595179, 0.8324666903089932, 0.5694681003946414, -4.008956506861486, 1.765914700186229]], [[-2.7412712675088446, 3.023835745744592, -2.5302327317041344, 5.749988165791118, -1.3161353426065303, 2.677038769225739], [-5.577103435175897, 8.639941855742613, 2.8676383502246563, 5.028435343385994, -6.408101804443406, 2.3124021799764276], [4.458038363216801, 1.1051734123685635, 1.9947300663533103, 0.9350145805046034, -3.2787875390025096, 3.4556743046512164], [3.4277015601381553, -9.964605325031958, 6.611302505535707, 6.743705726606957, 0.7486973826961454, -0.6770744726502496], [6.99393622159343, 2.5173448265893015, 2.1517218068686663, -4.435502886521648, 1.2438795596854508, -2.626523298350266], [1.7484520711131635, 1.6996630860033948, -4.057437497156621, -5.298930674694686, 10.166499450131251, 2.7389967187723334]], [[-0.7077623971028468, -6.390303590892487, -3.777762494737685, 0.9662784532474546, -1.7744597186397382, 1.447252452088052], [-3.8350529468858467, 3.066967568891998, 2.5873288023120873, -2.0868860722765357, -3.180122573897111, -2.3969916534048186], [4.930951122937193, 7.277842074380702, 9.728135411315108, -3.5218468619190464, -0.7362029157541704, 1.945449609291681], [-3.746982272513635, 3.4310221531099665, 2.7642304065069006, 0.5593102116382815, 1.7483411539784424, -3.423976314872309], [1.0770175523216519, -3.4343897065769085, -0.8612782049037527, 0.36124418915524176, 0.8020647332702708, 1.3448451400349073], [1.68618509948256, -4.844219353958285, 0.516296790485322, 3.1453541670349376, -1.9668591770195083, -0.17655002024169558]], [[1.7733568339342842, 4.305888651995983, 0.9220495153325785, 0.9484605827046637, 4.252609236767378, -0.017546116969284098], [0.555374186096113, -3.5421905829231335, -7.243567702650814, -6.353850777315959, -3.9112827671274815, -0.7237582792379109], [-1.1961886714296885, 7.030347800278448, -4.518642441530541, -1.51369946211266, -2.5913025023500937, -5.458793320773135], [-1.8294592613403207, -2.380659969954776, -1.7083948822916715, -10.333463051025404, -12.243481048558518, -4.993928122384068], [-7.410025161775432, 2.2867950437338553, -5.8747099998991255, 1.8224180609821217, 2.947743821533094, -3.792209226945153], [-7.526004803653359, 0.7291506228217657, 0.5550783642924328, -0.9751198099981444, 0.13481750373544465, -0.001385822598974018]]]], "dtype": "float64"}, {"__nd__": [[[[-3.3375519122331347, 5.019715109354992, -6.533228410219328, -1.0213033445401385, 9.047998587189014, -1.4033392457796678], [-5.462688236103448, -0.72295210918581, 4.960405905549165, 1.0449635430650708, 9.99672137055812, -6.841215583403065], [-1.805468592569268, 5.305612460331876, 0.5174414873305627, -10.512162712412305, -0.7728878635312384, 10.133247675505883], [-0.6075426195565031, -3.58238432350997, -4.500861697245853, 13.382983670337959, -6.450177974777851, 4.3816053081157715], [-6.2873165009694, 4.622479719651004, -3.072261349898404, -7.679050711438631, 9.191577315354698, -4.592782682744695], [3.5310342235591214, -6.277339940348481, 1.1051693116610186, 2.2314927746350044, 2.0372139753241423, -9.969027290657696]], [[-5.685253488087007, -10.594602355648515, -18.680076638823643, -9.730854420709582, 3.1985211220697427, -1.0729842190869667], [10.383586452662012, -4.559988100986228, -4.500264994750797, 10.183571993399818, 7.4958393237515315, -0.1741252046688908], [-2.2915497077910145, -7.398198023961708, -0.7845760198785966, 5.4618358963928, -2.797287534991594, -5.578074422261246], [1.6010934575053866, 23.07058652022418, 6.332247126926423, 5.116430789452095, 1.356076766295259, -3.4585670216652566], [0.9025884213525185, 2.2909376911643546, -9.72413667508493, -2.1528910547187, 3.6596415107178983, -3.239804286843127], [3.599225112068477, -0.7779774533352402, -7.208911333350245, 7.934710691499651, 1.706341181403839, -5.530062436514313]]], [[[11.22455863074597, -6.569914515369378, 2.5320353263618487, -2.090240453340495, -5.184667944674634, 4.325997181800116], [1.7055144720144115, 1.8988065582432658, 1.7038614377594987, 3.4575869387022227, -0.5102304031606636, -6.253975045120862], [-4.583381024132954, 8.82606091325914, -1.79131050033493, -7.839382773434232, -6.428636957670112, -5.710908879181581], [-0.5326551668186897, -3.341405217360895, 10.590859581686878, -9.112833493404437, 0.4336830211448479, 2.8991078298869173], [-7.7669821416088105, 6.694035964959572, -9.024473698092713, 12.501623144052708, -10.783025583308579, -2.4371212080860643], [4.107776031814723, -4.769299312405979, 2.4303209433610897, -1.0678280737846673, -4.864077808971462, -2.3905352088949527]], [[4.183907623072086, 1.2849018906758045, -1.2892149887795066, -2.5552410539906396, 13.70750488352712, 1.7374974757058155], [-1.2281462223876982, 2.3975232102177704, -8.83102034749558, -6.69234079844057, 2.0394683797140765, -5.575699023461102], [-0.5037362521496714, -0.08050185235960261, 2.642340413831793, 0.14038492537387437, -4.565167574984851, -7.8805496612791055], [-3.3651902175844475, -0.5452154506772133, -0.9405945980996004, 4.1545268829605675, 1.6235217972590295, -7.923557731017366], [-0.03845487680250659, -6.960098544652274, -2.1897157824819837, 5.102403777527579, 1.6869054690810932, -4.01723334130487], [-1.8358632775932109, -0.18389251214780333, 1.5142352391413099, 3.8139372047534743, 5.336642242925622, 0.5874000817281471]]], [[[-2.9115588987927503, 1.1999726312833447, -5.664982254695268, -6.096947650993349, 7.771523321547796, -1.6056016637718251], [2.1579705871505226, 1.1983798502826848, -7.000711703108131, 3.87833077723561, -2.9764295505142226, -9.690830891993386], [1.078651535727133, -6.569108364425888, -0.32607029903364504, -8.280383179452798, -15.411824788986225, 10.587559820061891], [-0.8169675814299648, 9.689710564602136, -8.430420454075934, -9.633782465402428, 7.22409772754421, 2.7378270878614224], [10.29183268066959, -14.554027336149277, 9.82226151370826, 0.3078413622317733, 4.531305727359329, -13.875479176656413], [-3.6439699155579293, 5.272274604887583, 2.479594086037985, -5.270975045399995, -8.219798598131195, -4.722146335708202]], [[5.965971928070951, 8.500822189170401, 8.501980292453197, -0.7470502516192932, 4.491528137156468, 2.7370088178187695], [-8.769594525253172, 3.782557480309976, -13.768532064773114, -7.367899421168097, 10.034867429702134, 1.0608726062484968], [5.743205827864408, 0.3838891288605524, 3.1420040399186817, 7.429373062808643, 9.909779243971709, 3.050076719585378], [-15.292888611600597, -21.22057013613762, -14.163808704479079, -7.282750022544722, 0.7622965480951333, -0.48579469658402447], [3.7888440923339046, 6.890009029173575, 7.7102258116799804, 8.57510222773978, 1.1128663758048776, 2.964187923212656], [-0.8984867352880642, -1.0206485036348396, -6.621411920611299, -7.733328645396803, -4.672137204258642, -2.299279417918197]]]], "dtype": "float64"}, {"__nd__": [[[[1.4642634993958563, 1.4705955679685898, 8.845692521269356, -0.10600723685531044, -2.5338766782073887], [-8.009570700715892, -2.27113523152825, 1.3443949282252294, -9.251177224781202, -7.165065906382853], [5.561368046199193, -16.117518641663228, -17.148452166702423, -5.339134752271306, 3.4240688931931476], [7.782963599304113, 5.696442594206394, -12.090613153841348, 4.955982892785407, 2.945277198126351], [-2.555767733229102, 5.592777934637493, 16.0670202428536, -1.6787739586403148, 0.6971624228627138]]], [[[13.434294868694785, -7.718092924932827, -0.6799256082197367, 10.344979740331118, -1.0161572472042495], [-2.399264523283451, -13.121027372838519, -7.807589946519279, -0.1366416270715174, 3.5257912149376596], [3.978246604456177, -5.153987023778385, -9.087773952924094, -12.442396903546252, 13.990032512541694], [1.7695458165569757, -8.40737354026506, -3.492148367198218, -6.496155254665019, -3.2745425355759954], [-14.71544027973457, -10.563535729380469, 6.70290875212238, -1.8670417049441386, -8.27616718093195]]], [[[-6.553408598658336, 2.210611759337878, -15.061547256565625, -0.5052226621717069, -2.209485603166126], [14.031557327548244, 2.0889465449795805, 8.176991024672947, 1.1983095117196312, 0.4487893992132004], [-3.4180354732804155, -1.8026781724124987, 4.476016858009579, -7.909789376722241, -2.3640778987508027], [8.67855558841884, -0.5890950166195719, -17.89097380131742, 18.472690096191265, -2.6929817363705295], [-12.793024140597083, -2.573136976591973, 8.526588606384701, -14.40155803577089, 13.040054249538516]]], [[[-11.174378457768096, 1.7478732779759207, 10.911866236887072, 3.6711074982492833, -6.0565023352559635], [0.9376455843390681, 2.0355500627227725, 14.14759465823922, -4.848722136738978, -9.869252107599028], [-16.84490364509821, 12.66677289025469, -3.4352474954264127, -6.421015269244059, 12.251612573737857], [-7.140142768335248, 2.6912738543396513, 3.7200722987009027, -8.48252350207093, 1.7022683369142504], [-13.3148138494894, 14.965867858070506, -4.087071048593123, -17.351018676249687, 4.585236562524719]]]], "dtype": "float64"}, {"__nd__": [14.571358558339679, 8.692080123254218, -4.481696848435659, 17.43577960178307], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(conv2d_unfold, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")